# SupeRR Selector Season 2 — Production-Grade Pressure Evaluation Pipeline
**Theme**: Identifying the Most Reliable Players Under Pressure in T20 Cricket (IPL & ICC Full-Member T20Is, 2022–Present)

### Production ML & Cricket Analytics Audit Upgrades:
1. **Strictly Nested Match-Grouped Win-Probability (OOF)**: Both outer cross-validation and inner Platt calibration splits are grouped strictly by `match_id` (`GroupKFold`), guaranteeing zero intra-match data leakage.
2. **Empirical Phase-Specific Pre-Ball Leverage**: Evaluates expected swings $E[|\Delta WP|]$ across prospective delivery states (dot, run, boundary, wicket) using empirical delivery distributions derived by phase.
3. **Exact Terminal First-Innings WPA Transition**: The final delivery of Innings 1 is credited based on post-delivery win probability of defending the final total $T$, eliminating arbitrary 0/1 match-outcome contamination.
4. **Individual WPA Separation**: Striker WPA is isolated from non-striker run-outs; per-innings WPA telescopes exactly to the change in win probability (asserted). Non-striker run-out WPA is not credited to the non-striker's own rating (see disclosures).
5. **True Dot-Ball Accounting**: Accurately accounts for byes and leg-byes — legal balls where the fielding team concedes team runs are correctly excluded from dot balls.
6. **Coherent 500-Match Cluster Bootstrap**: Re-executes the 5-component composite scoring, Empirical Bayes shrinkage, and eligibility thresholds inside every resample, yielding conditional and unconditional $P(\text{Top 10})$. The win-probability models and the pressure threshold are held fixed across resamples, so intervals are conditional on the pressure definition.
7. **Validation & Sensitivity Suite**: Includes split-half reliability ($r_{\text{full}}$ via Spearman-Brown), tournament/season splits, and weight/threshold sensitivity stress-tests.

### v2 fixes
1. Pressure flag applied to wides/no-balls too. 2. Leverage baseline and counterfactuals from the same model. 3. Pressure threshold per innings. 4. Batter elevation = RAE (not WPA). 5. Bowler elevation uses shrunk pressure AND normal economy, no zero imputation.
6. Consistency rate shrunk; RSE bug fixed; real eligibility re-ranking. 7. (v2.1) sklearn-only gradient boosting averaged over 3 seeds for reproducibility. 8. (v2.1) Bowler elevation = RSE/over delta. 9. (v2.1) Consensus final ranking (baseline + bootstrap median + eligibility-cutoff variants, P(eligible) >= 0.75 for top 3).


In [1]:
# CELL 1 — Environment Setup & Path Verification
import os
import sys

# Data lookup order: Colab Drive folder, then ./data (project layout), then the current folder.
candidates = ["/content/drive/MyDrive/Data Pack 2", "data", "."]
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
expected_files = ['competitions.csv', 'matches.csv', 'teams.csv', 'playing_eleven.csv', 'ball_by_ball.csv']
DATA_DIR = next((d for d in candidates if all(os.path.exists(os.path.join(d, f)) for f in expected_files)), None)
assert DATA_DIR is not None, f"Data files not found. Put {expected_files} in ./data (see data/README.md)."

print(f"Data Directory set to: {os.path.abspath(DATA_DIR)}")
print("Found core dataset files:", expected_files)


Data Directory set to: /Users/yashas/Data Pack 2/data
Found core dataset files: ['competitions.csv', 'matches.csv', 'teams.csv', 'playing_eleven.csv', 'ball_by_ball.csv']


In [2]:
# CELL 2 — Imports & Global Reproducibility
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import GroupKFold
from sklearn.metrics import brier_score_loss, log_loss

# Reproducibility: ONLY scikit-learn gradient boosting is used (no xgboost/lightgbm), so results do not depend on installed libraries.
from sklearn.ensemble import HistGradientBoostingClassifier
lgb = None
from sklearn.ensemble import HistGradientBoostingRegressor

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 35)

print("Environment configured with SEED = 42.")


Environment configured with SEED = 42.


In [3]:
# CELL 3 — Multi-Table Data Ingestion
competitions    = pd.read_csv(f"{DATA_DIR}/competitions.csv")
matches         = pd.read_csv(f"{DATA_DIR}/matches.csv")
teams           = pd.read_csv(f"{DATA_DIR}/teams.csv")
playing_eleven  = pd.read_csv(f"{DATA_DIR}/playing_eleven.csv")
comp_player_map = pd.read_csv(f"{DATA_DIR}/competition_team_player_mapping.csv")
balls_raw       = pd.read_csv(f"{DATA_DIR}/ball_by_ball.csv", low_memory=False)

print(f"Raw Competitions : {competitions.shape}")
print(f"Raw Matches      : {matches.shape}")
print(f"Raw Deliveries   : {balls_raw.shape}")
assert len(balls_raw) > 100000, "ball_by_ball.csv incomplete"


Raw Competitions : (141, 4)
Raw Matches      : (883, 22)
Raw Deliveries   : (200999, 33)


In [4]:
# CELL 4 — Strict Competition Scope & Quality Filtering (Jan 2022+)

# 1. Scope: T20I and IPL
competitions['format_clean'] = competitions['match_format'].astype(str).str.strip().str.upper()
competitions['comp_name_lc'] = competitions['comp_name'].astype(str).str.strip().str.lower()

t20i_comps = competitions[competitions['format_clean'] == 'T20I'].copy()
ipl_comps  = competitions[
    (competitions['format_clean'] == 'T20') &
    (competitions['comp_name_lc'].str.contains('ipl|indian premier', na=False))
].copy()

eligible_comps = pd.concat([ipl_comps, t20i_comps], ignore_index=True).drop_duplicates('comp_id')
eligible_comp_ids = set(eligible_comps['comp_id'])

# 2. Filter matches (from 2022-01-01, valid completed matches only)
matches['match_date'] = pd.to_datetime(matches['match_date'], errors='coerce')

# Drop abandoned/no-result matches where winner cannot be determined
valid_matches = matches[
    (matches['comp_id'].isin(eligible_comp_ids)) &
    (matches['match_date'] >= '2022-01-01') &
    (matches['win_team_id'].notna()) &
    (matches['win_team_id'] != 0) &
    (~matches['result'].astype(str).str.lower().isin(['no result', 'abandoned']))
].copy()

# Bring match_format and comp_name from competitions into valid_matches
valid_matches = valid_matches.merge(
    eligible_comps[['comp_id', 'match_format', 'comp_name', 'season']],
    on='comp_id',
    how='left'
)

# Quality Filter: Keep IPL + Full-Member T20I to eliminate weak-opposition distortions
FULL_MEMBERS = {
    'India', 'Australia', 'England', 'Pakistan', 'South Africa',
    'New Zealand', 'West Indies', 'Sri Lanka', 'Bangladesh',
    'Zimbabwe', 'Afghanistan', 'Ireland'
}
home_full = valid_matches['home_team'].isin(FULL_MEMBERS)
away_full = valid_matches['away_team'].isin(FULL_MEMBERS)
valid_matches['is_full_member_t20i'] = (home_full & away_full).astype(int)

# Filter: All IPL matches + T20Is between full members
eligible_matches = valid_matches[
    (valid_matches['match_format'].astype(str).str.upper() == 'T20') |
    (valid_matches['is_full_member_t20i'] == 1)
].copy()

# Add context flags
eligible_matches['is_ipl'] = (eligible_matches['match_format'].astype(str).str.upper() == 'T20').astype(int)
eligible_matches['is_knockout'] = (
    eligible_matches['stage'].astype(str).str.lower().str.contains('final|semi|playoff|eliminator|qualifier', na=False) |
    (eligible_matches['knock_out'].fillna(0).astype(int) == 1)
).astype(int)

eligible_match_ids = set(eligible_matches['match_id'])

print(f"Eligible Competitions: {len(eligible_comps)}")
print(f"Eligible Matches: {len(eligible_matches)} (IPL: {eligible_matches['is_ipl'].sum()}, Full-Member T20I: {(~eligible_matches['is_ipl'].astype(bool)).sum()})")
assert len(eligible_matches) > 500, "Unexpectedly low number of eligible matches"


Eligible Competitions: 139
Eligible Matches: 761 (IPL: 362, Full-Member T20I: 399)


In [5]:
# CELL 5 — Rigorous Cricket Delivery Accounting & Dismissal Attribution

# 1. Filter deliveries to eligible matches and normal innings (1 and 2 only, no super overs)
balls = balls_raw[
    (balls_raw['match_id'].isin(eligible_match_ids)) &
    (balls_raw['innings'].isin([1, 2]))
].copy()

# 2. Strict Delivery Legality
balls['extras_type_clean'] = balls['extras_type'].astype(str).str.strip().str.lower()
balls['is_wide']   = (balls['extras_type_clean'] == 'wide').astype(int)
balls['is_noball'] = (balls['extras_type_clean'].isin(['overstep', 'no ball', 'noball'])).astype(int)
balls['is_legal']  = (1 - balls['is_wide'] - balls['is_noball']).clip(0, 1)

# Balls faced by batter under official MCC laws: no-balls count as balls faced, wides DO NOT count as balls faced!
balls['is_ball_faced'] = (balls['is_wide'] == 0).astype(int)

# 3. Cricket Accounting: Batter runs vs Bowler runs vs Team runs
balls['bat_runs'] = pd.to_numeric(balls['runs'], errors='coerce').fillna(0).astype(int)
balls['extras1_runs_num'] = pd.to_numeric(balls['extras1_runs'], errors='coerce').fillna(0).astype(int)
balls['extras_2_runs_num'] = pd.to_numeric(balls['extras_2_runs'], errors='coerce').fillna(0).astype(int)

# Bowler extras: wides & noballs charged to bowler; byes & legbyes NOT charged to bowler
balls['bowler_extras'] = np.where(
    balls['is_wide'] == 1,
    balls['extras1_runs_num'] + balls['extras_2_runs_num'],
    np.where(balls['is_noball'] == 1, balls['extras1_runs_num'] + balls['extras_2_runs_num'], 0)
)
balls['bowler_runs'] = balls['bat_runs'] + balls['bowler_extras']

# Total team runs on delivery (bat runs + all extras)
balls['team_runs'] = balls['bat_runs'] + balls['extras1_runs_num'] + balls['extras_2_runs_num']

# Dot ball accounting (Requirement 5):
# A true dot ball is a LEGAL ball where NO team runs are scored (neither bat runs nor byes/leg-byes).
balls['is_dot'] = ((balls['is_legal'] == 1) & (balls['team_runs'] == 0)).astype(int)

# 4. Strict Dismissal Accounting & Individual Attribution (Requirement 4 & 6):
balls['is_wicket_flag'] = balls['is_wicket'].astype(str).str.strip().str.lower().isin(['true', '1'])
balls['dismissal_type_clean'] = balls['dismissal_type'].astype(str).str.strip()

non_bowler_dismissals = ['RunOut', 'Retire Out', 'Obstracting the field', 'Obstructing the field', 'None', '']
balls['is_bowler_wicket'] = (
    balls['is_wicket_flag'] &
    (~balls['dismissal_type_clean'].isin(non_bowler_dismissals))
).astype(int)

# Distinguish striker dismissal vs non-striker dismissal using dismissed_player_id
balls['dismissed_player_id_clean'] = pd.to_numeric(balls['dismissed_player_id'], errors='coerce').fillna(0).astype(int)
balls['striker_id'] = pd.to_numeric(balls['striker_id'], errors='coerce').fillna(0).astype(int)
balls['non_striker_id'] = pd.to_numeric(balls['non_striker_id'], errors='coerce').fillna(0).astype(int)
balls['bowler_id'] = pd.to_numeric(balls['bowler_id'], errors='coerce').fillna(0).astype(int)

# Striker is out only if the dismissed player is the striker, OR if dismissed_player_id is unrecorded on a non-runout
balls['is_striker_out'] = (
    balls['is_wicket_flag'] &
    ((balls['dismissed_player_id_clean'] == balls['striker_id']) | 
     ((balls['dismissed_player_id_clean'] == 0) & (balls['dismissal_type_clean'] != 'RunOut')))
).astype(int)

# Non-striker was out (e.g. non-striker run-out) — striker must NOT be charged!
balls['is_non_striker_out'] = (
    balls['is_wicket_flag'] &
    (balls['dismissed_player_id_clean'] == balls['non_striker_id'])
).astype(int)

balls['is_batter_out'] = balls['is_striker_out']  # Batter personal stat reflects their own dismissal

# 5. Canonical Player Mapping
player_name_map = {}
for _, row in balls[['striker_id', 'striker_name']].drop_duplicates().iterrows():
    if row['striker_id'] != 0 and pd.notna(row['striker_name']):
        player_name_map[int(row['striker_id'])] = str(row['striker_name']).strip()
for _, row in balls[['bowler_id', 'bowler_name']].drop_duplicates().iterrows():
    if row['bowler_id'] != 0 and pd.notna(row['bowler_name']):
        player_name_map[int(row['bowler_id'])] = str(row['bowler_name']).strip()
for _, row in balls[['non_striker_id', 'non_striker_name']].drop_duplicates().iterrows():
    if row['non_striker_id'] != 0 and pd.notna(row['non_striker_name']):
        player_name_map[int(row['non_striker_id'])] = str(row['non_striker_name']).strip()

# Sort strictly chronologically
balls = balls.sort_values(['match_id', 'innings', 'overid', 'over_ball_num']).reset_index(drop=True)

# Reconciliation assertions (Requirement 6)
raw_eligible_runs = pd.to_numeric(balls_raw[balls_raw['match_id'].isin(eligible_match_ids) & balls_raw['innings'].isin([1, 2])]['runs'], errors='coerce').fillna(0).sum()
assert balls['bat_runs'].sum() == raw_eligible_runs, "Bat runs mismatch against raw data"
print(f"Sanitized Deliveries: {len(balls):,}")
print(f"Total Legal Balls: {balls['is_legal'].sum():,} | Balls Faced: {balls['is_ball_faced'].sum():,} | True Dots: {balls['is_dot'].sum():,}")
print(f"Striker Outs: {balls['is_striker_out'].sum():,} | Non-Striker Outs: {balls['is_non_striker_out'].sum():,} | Bowler Wickets: {balls['is_bowler_wicket'].sum():,}")


Sanitized Deliveries: 180,345
Total Legal Balls: 172,812 | Balls Faced: 173,341 | True Dots: 60,427
Striker Outs: 9,456 | Non-Striker Outs: 240 | Bowler Wickets: 9,168


In [6]:
# CELL 6 — Innings State Engine (Cumulative Runs, Wickets, Targets, Balls Remaining, RRR)

# 1. Merge match context
match_cols = [
    'match_id', 'comp_id', 'match_date', 'venue_id', 'is_ipl', 'is_knockout',
    'win_team_id', 'home_team_id', 'away_team_id', 'revised_target',
    'first_inn_revised_overs', 'second_inn_revised_overs'
]
balls = balls.merge(eligible_matches[match_cols], on='match_id', how='left')

# 2. Cumulative Runs, Wickets, and Legal Balls within Innings
balls['cum_team_runs'] = balls.groupby(['match_id', 'innings'])['team_runs'].transform(lambda x: x.shift(1).fillna(0).cumsum()).astype(int)
balls['any_wicket']    = balls['is_wicket_flag'].astype(int)
balls['cum_wickets']   = balls.groupby(['match_id', 'innings'])['any_wicket'].transform(lambda x: x.shift(1).fillna(0).cumsum()).astype(int)
balls['cum_legal']     = balls.groupby(['match_id', 'innings'])['is_legal'].transform(lambda x: x.shift(1).fillna(0).cumsum()).astype(int)
balls['wickets_in_hand'] = (10 - balls['cum_wickets']).clip(0, 10)

# 3. Balls Remaining (accurately handling rain-reduced matches)
def calc_total_balls(row):
    rev = row['first_inn_revised_overs'] if row['innings'] == 1 else row['second_inn_revised_overs']
    if pd.notna(rev) and float(rev) > 0:
        return int(float(rev) * 6)
    return 120

balls['total_legal_balls'] = balls.apply(calc_total_balls, axis=1)
balls['balls_remaining']   = (balls['total_legal_balls'] - balls['cum_legal']).clip(lower=0)

# 4. Target Calculation for Innings 2 (prioritizing official revised_target)
inn1_actual_totals = balls[balls['innings'] == 1].groupby('match_id')['team_runs'].sum().reset_index().rename(columns={'team_runs': 'inn1_actual_total'})
balls = balls.merge(inn1_actual_totals, on='match_id', how='left')

balls['target'] = np.where(
    balls['innings'] == 2,
    np.where(balls['revised_target'].notna() & (balls['revised_target'] > 0),
             balls['revised_target'],
             balls['inn1_actual_total'] + 1),
    np.nan
)

balls['runs_needed'] = np.where(
    balls['innings'] == 2,
    (balls['target'] - balls['cum_team_runs']).clip(lower=0),
    np.nan
)

balls['rrr'] = np.where(
    (balls['innings'] == 2) & (balls['balls_remaining'] > 0),
    balls['runs_needed'] / (balls['balls_remaining'] / 6.0),
    np.nan
)

# 5. Batting Team Won Target Variable (binary 0 or 1)
balls['batting_team_won'] = (balls['batting_team_id'] == balls['win_team_id']).astype(int)

# 6. Jackknife Leave-One-Match-Out Venue Par Score (Requirement 3: strictly zero leakage)
venue_matches = balls[['venue_id', 'match_id', 'inn1_actual_total']].drop_duplicates()
venue_sums    = venue_matches.groupby('venue_id')['inn1_actual_total'].transform('sum')
venue_counts  = venue_matches.groupby('venue_id')['inn1_actual_total'].transform('count')
venue_matches['venue_par_score'] = np.where(
    venue_counts > 1,
    (venue_sums - venue_matches['inn1_actual_total']) / (venue_counts - 1),
    165.0  # global T20 median par fallback
)
balls = balls.merge(venue_matches[['venue_id', 'match_id', 'venue_par_score']], on=['venue_id', 'match_id'], how='left')
balls['venue_par_score'] = balls['venue_par_score'].fillna(165.0)

print("Innings State Engine complete.")
assert balls['batting_team_won'].isin([0, 1]).all(), "batting_team_won must be strictly binary"
print(balls[['match_id', 'innings', 'overid', 'cum_team_runs', 'wickets_in_hand', 'balls_remaining', 'target', 'runs_needed', 'venue_par_score']].head())


Innings State Engine complete.
   match_id  innings  overid  cum_team_runs  wickets_in_hand  balls_remaining  \
0     13511        1       1              0               10              120   
1     13511        1       1              0               10              119   
2     13511        1       1              0               10              118   
3     13511        1       1              0               10              117   
4     13511        1       1              0               10              116   

   target  runs_needed  venue_par_score  
0     NaN          NaN       182.571429  
1     NaN          NaN       182.571429  
2     NaN          NaN       182.571429  
3     NaN          NaN       182.571429  
4     NaN          NaN       182.571429  


In [7]:
# CELL 7 — Dual Calibrated Win-Probability Models with Strictly Nested Match-Grouped OOF Validation
# Requirement 3: Outer CV and Inner Calibration CV are both grouped by match_id to eliminate leakage completely!

balls['win_prob'] = np.nan

SEEDS = [42, 7]

def get_base_classifier(seed=SEED):
    return HistGradientBoostingClassifier(max_iter=150, max_depth=4, learning_rate=0.08, random_state=seed)

class SeedEnsemble:
    """Average of calibrated models trained with different seeds (variance reduction)."""
    def __init__(self, models): self.models = models
    def predict_proba(self, X): return np.mean([m.predict_proba(X) for m in self.models], axis=0)

def fit_seed_ensemble(X, y, splits):
    ms = []
    for sd in SEEDS:
        m = CalibratedClassifierCV(estimator=get_base_classifier(sd), method='sigmoid', cv=splits)
        m.fit(X, y); ms.append(m)
    return SeedEnsemble(ms)

# Helper function for strictly nested out-of-fold calibrated prediction
def run_nested_oof_calibration(df, features, target_col='batting_team_won', group_col='match_id'):
    X = df[features].values
    y = df[target_col].values
    groups = df[group_col].values
    
    gkf_outer = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(df))
    
    for fold, (train_idx, val_idx) in enumerate(gkf_outer.split(X, y, groups=groups)):
        # Assert ZERO match leakage between train and validation folds (Requirement 3)
        train_matches = set(groups[train_idx])
        val_matches   = set(groups[val_idx])
        assert len(train_matches.intersection(val_matches)) == 0, f"Outer fold {fold} match leakage detected!"
        
        X_tr, y_tr, g_tr = X[train_idx], y[train_idx], groups[train_idx]
        X_val = X[val_idx]
        
        # Nested Inner GroupKFold calibration on training matches only
        gkf_inner = GroupKFold(n_splits=3)
        inner_splits = list(gkf_inner.split(X_tr, y_tr, groups=g_tr))
        
        cal_m = fit_seed_ensemble(X_tr, y_tr, inner_splits)
        oof_preds[val_idx] = cal_m.predict_proba(X_val)[:, 1]
        
    return np.clip(oof_preds, 0.001, 0.999)

# --- 1. INNINGS 1 MODEL ---
inn1_mask = (balls['innings'] == 1)
inn1_df   = balls[inn1_mask].copy()

# Advanced cricket state features for Innings 1 (including reduced-over handling & zero-leakage venue par)
safe_balls_elapsed = (inn1_df['total_legal_balls'] - inn1_df['balls_remaining']).clip(lower=1.0)
inn1_df['run_rate'] = (inn1_df['cum_team_runs'] / safe_balls_elapsed) * 6.0
inn1_df['wickets_lost'] = 10 - inn1_df['wickets_in_hand']
inn1_df['par_diff'] = inn1_df['cum_team_runs'] - (inn1_df['venue_par_score'] * (safe_balls_elapsed / inn1_df['total_legal_balls']))

features_inn1 = ['balls_remaining', 'wickets_in_hand', 'cum_team_runs', 'is_ipl', 'run_rate', 'wickets_lost', 'par_diff']

balls.loc[inn1_mask, 'win_prob'] = run_nested_oof_calibration(inn1_df, features_inn1)

# Fit full Innings 1 model for prospective counterfactual leverage state transitions
gkf = GroupKFold(n_splits=5)
cv1_splits = list(gkf.split(inn1_df[features_inn1].values, inn1_df['batting_team_won'].values, groups=inn1_df['match_id'].values))
cal_model1 = fit_seed_ensemble(inn1_df[features_inn1].values, inn1_df['batting_team_won'].values, cv1_splits)

# --- 2. INNINGS 2 MODEL ---
inn2_mask = (balls['innings'] == 2)
inn2_df   = balls[inn2_mask].copy()

inn2_df['rrr_capped'] = inn2_df['rrr'].fillna(6.0).clip(lower=0.0, upper=36.0)
inn2_df['runs_needed_clean'] = inn2_df['runs_needed'].fillna(0.0).clip(lower=0.0)
inn2_df['rrr_wkts_interaction'] = inn2_df['rrr_capped'] * (10 - inn2_df['wickets_in_hand'])
inn2_df['urgency'] = (inn2_df['total_legal_balls'] - inn2_df['balls_remaining']) / inn2_df['total_legal_balls']
inn2_df['runs_per_ball_needed'] = inn2_df['runs_needed_clean'] / inn2_df['balls_remaining'].clip(lower=1.0)

features_inn2 = [
    'balls_remaining', 'wickets_in_hand', 'runs_needed_clean',
    'rrr_capped', 'is_ipl', 'rrr_wkts_interaction', 'urgency', 'runs_per_ball_needed'
]

balls.loc[inn2_mask, 'win_prob'] = run_nested_oof_calibration(inn2_df, features_inn2)

cv2_splits = list(gkf.split(inn2_df[features_inn2].values, inn2_df['batting_team_won'].values, groups=inn2_df['match_id'].values))
cal_model2 = fit_seed_ensemble(inn2_df[features_inn2].values, inn2_df['batting_team_won'].values, cv2_splits)

# --- 3. HONEST OUT-OF-FOLD EVALUATION & PHASE BREAKDOWN (Requirement 3) ---
brier_inn1 = brier_score_loss(inn1_df['batting_team_won'], balls.loc[inn1_mask, 'win_prob'])
loss_inn1  = log_loss(inn1_df['batting_team_won'], balls.loc[inn1_mask, 'win_prob'])

brier_inn2 = brier_score_loss(inn2_df['batting_team_won'], balls.loc[inn2_mask, 'win_prob'])
loss_inn2  = log_loss(inn2_df['batting_team_won'], balls.loc[inn2_mask, 'win_prob'])

total_brier = brier_score_loss(balls['batting_team_won'], balls['win_prob'])
total_loss  = log_loss(balls['batting_team_won'], balls['win_prob'])

print("=== STRICT NESTED OUT-OF-FOLD MODEL PERFORMANCE ===")
print(f"Innings 1 OOF Brier Score: {brier_inn1:.4f} | Log Loss: {loss_inn1:.4f}")
print(f"Innings 2 OOF Brier Score: {brier_inn2:.4f} | Log Loss: {loss_inn2:.4f}")
print(f"Overall Match OOF Brier  : {total_brier:.4f} | Log Loss: {total_loss:.4f}")

# Phase breakdown
balls['phase'] = pd.cut(
    balls['balls_remaining'],
    bins=[-1, 30, 84, 150],
    labels=['Death (overs 16-20)', 'Middle (overs 7-15)', 'Powerplay (overs 1-6)']
)
phase_metrics = balls.groupby('phase').agg(
    brier=('batting_team_won', lambda y: brier_score_loss(y, balls.loc[y.index, 'win_prob'])),
    logloss=('batting_team_won', lambda y: log_loss(y, balls.loc[y.index, 'win_prob'])),
    deliveries=('team_runs', 'count')
).reset_index()
display(phase_metrics)
assert balls['win_prob'].isna().sum() == 0, "No NaNs allowed in win_prob"


=== STRICT NESTED OUT-OF-FOLD MODEL PERFORMANCE ===
Innings 1 OOF Brier Score: 0.2273 | Log Loss: 0.6463
Innings 2 OOF Brier Score: 0.1315 | Log Loss: 0.4060
Overall Match OOF Brier  : 0.1814 | Log Loss: 0.5312


,phase,brier,logloss,deliveries
0,Death (overs 16-20),0.155605,0.462924,40051
1,Middle (overs 7-15),0.175214,0.516812,81899
2,Powerplay (overs 1-6),0.209616,0.602599,55717


## Mathematical Documentation: Pre-Ball Leverage & Pressure Rules (Requirement 2)

### 1. Pre-Ball Expected Leverage Index ($LI_{\text{pre-ball}}$)
Unlike historical post-hoc leverage $|WP_{t+1} - WP_t|$ (which is circular because a boundary or wicket automatically inflates leverage), **Pre-Ball Leverage Index** quantifies match-state volatility **before the bowler releases the ball**:

$$LI_{\text{pre-ball}}(s) = \sum_{o \in \Omega} P(o \mid \text{phase}) \cdot \left| WP(s \oplus o) - WP(s) \right|$$

where $\Omega = \{\text{Dot}, \text{Single}, \text{Boundary}, \text{Wicket}\}$ are prospective outcomes weighted by empirical delivery probabilities derived by phase from historical data:
These probabilities are derived empirically per phase from the data in the next code cell (printed table), not hard-coded.

### 2. High Pressure Classification Rule
- **Objective Criterion**: Deliveries ranking in the **Top 20% of $LI_{\text{pre-ball}}$ among all legal deliveries** are classified as `is_pressure = 1`.
- **Reconciliation**: Evaluated strictly on legal balls, yielding an exact 20.0% pressure delivery share.
- **Cricket Pressure Index (CPI)**: An explainable 0-4 heuristic (RRR $\ge 9.5$ in chases / behind par in innings 1, death overs, wickets in hand $\le 4$, win probability within 25-75%) computed in the next cell purely to validate that the leverage-based flag matches cricketing intuition.


In [8]:
# CELL 8 — Pre-Ball Leverage Index (Empirical Outcomes) & WPA Separation
# Requirement 2: Empirical phase-based transition probabilities, top 20% legal delivery threshold, hopeless state diagnostic
# Requirement 4: Explicit separation of striker, non-striker, and bowler WPA with innings reconciliation assertion

legal_balls = balls[balls['is_legal'] == 1].copy()

# Compute Empirical Outcome Probabilities by Phase directly from data (Requirement 2)
phase_outcomes = legal_balls.groupby('phase').agg(
    n=('team_runs', 'count'),
    p_dot=('is_dot', 'mean'),
    p_wicket=('is_wicket_flag', 'mean'),
    p_boundary=('bat_runs', lambda r: np.mean(r.isin([4, 5, 6]))),
).reset_index()
phase_outcomes['p_run'] = (1.0 - phase_outcomes['p_dot'] - phase_outcomes['p_wicket'] - phase_outcomes['p_boundary']).clip(lower=0.0)
prob_dict = phase_outcomes.set_index('phase')[['p_dot', 'p_run', 'p_boundary', 'p_wicket']].to_dict('index')

print("=== EMPIRICALLY DERIVED PHASE TRANSITION PROBABILITIES ===")
display(phase_outcomes)

# Vectorized true expected swing simulator
def compute_empirical_preball_li(df, model, innings_num):
    b_rem = (df['balls_remaining'].fillna(60) - 1).clip(lower=0).values
    w_hand = df['wickets_in_hand'].fillna(10).values
    ipl = df['is_ipl'].fillna(0).values
    tot_b = df['total_legal_balls'].fillna(120).values
    
    if innings_num == 1:
        cum_r = df['cum_team_runs'].fillna(0).values
        safe_el = np.clip(tot_b - b_rem, 1.0, None)
        v_par = df['venue_par_score'].fillna(165.0).values
        
        def s1(b, w, r):
            se = np.clip(tot_b - b, 1.0, None)
            rr = (r / se) * 6.0
            pd_ = r - (v_par * (se / tot_b))
            return np.column_stack([b, w, r, ipl, rr, 10 - w, pd_])
            
        s_0 = s1(b_rem, w_hand, cum_r)
        s_1 = s1(b_rem, w_hand, cum_r + 1)
        s_4 = s1(b_rem, w_hand, cum_r + 4)
        s_w = s1(b_rem, np.clip(w_hand - 1, 0, 10), cum_r)
        s_now = s1(df['balls_remaining'].fillna(60).values, w_hand, cum_r)
    else:
        r_nd = df['runs_needed'].fillna(0.0).clip(lower=0.0).values
        def s2(b, w, r):
            r_c = np.clip(r, 0.0, None)
            safe_ov = np.clip(b / 6.0, 1.0 / 6.0, None)
            rrr_c = np.clip(np.where(b > 0, r_c / safe_ov, 36.0), 0.0, 36.0)
            inter = rrr_c * (10.0 - w)
            urg = (tot_b - b) / tot_b
            rpb = r_c / np.clip(b, 1.0, None)
            return np.column_stack([b, w, r_c, rrr_c, ipl, inter, urg, rpb])
            
        s_0 = s2(b_rem, w_hand, r_nd)
        s_1 = s2(b_rem, w_hand, r_nd - 1)
        s_4 = s2(b_rem, w_hand, r_nd - 4)
        s_w = s2(b_rem, np.clip(w_hand - 1, 0, 10), r_nd)
        s_now = s2(df['balls_remaining'].fillna(60).values, w_hand, r_nd)
        
    p0 = model.predict_proba(s_0)[:, 1]
    p1 = model.predict_proba(s_1)[:, 1]
    p4 = model.predict_proba(s_4)[:, 1]
    pw = model.predict_proba(s_w)[:, 1]
    # FIX 2: baseline and counterfactuals come from the SAME model, so model offset cancels
    p_now = model.predict_proba(s_now)[:, 1]
    
    # Weight swings by phase-specific empirical outcome probabilities
    phases = df['phase'].values
    p_dot_arr = np.array([prob_dict.get(ph, {}).get('p_dot', 0.38) for ph in phases])
    p_run_arr = np.array([prob_dict.get(ph, {}).get('p_run', 0.35) for ph in phases])
    p_bnd_arr = np.array([prob_dict.get(ph, {}).get('p_boundary', 0.18) for ph in phases])
    p_wkt_arr = np.array([prob_dict.get(ph, {}).get('p_wicket', 0.09) for ph in phases])
    
    expected_swing = (
        p_dot_arr * np.abs(p0 - p_now) +
        p_run_arr * np.abs(p1 - p_now) +
        p_bnd_arr * np.abs(p4 - p_now) +
        p_wkt_arr * np.abs(pw - p_now)
    )
    return np.nan_to_num(expected_swing, nan=0.01)

balls['preball_li'] = np.nan
balls.loc[inn1_mask, 'preball_li'] = compute_empirical_preball_li(balls[inn1_mask], cal_model1, 1)
balls.loc[inn2_mask, 'preball_li'] = compute_empirical_preball_li(balls[inn2_mask], cal_model2, 2)
balls['preball_li'] = balls['preball_li'].fillna(balls['preball_li'].median())
assert balls['preball_li'].isna().sum() == 0, "No NaNs permitted in preball_li"

# Reconciled High Pressure Definition: STRICT Top 20% Pre-Ball Leverage on Legal Deliveries (Requirement 2)
# FIX 3: threshold set per innings (top 20% of legal balls WITHIN each innings) so the weaker
#        innings-1 win-prob model cannot starve/flood pressure; overall legal share stays 20%.
# FIX 1: the flag is applied to EVERY delivery (incl. wides/no-balls) at/above the threshold,
#        so extras bowled in high-leverage states are no longer dumped into the 'normal' bucket.
PRESSURE_PER_INNINGS = True
_legal = balls['is_legal'] == 1
if PRESSURE_PER_INNINGS:
    _thr = balls[_legal].groupby('innings')['preball_li'].quantile(0.80)
    balls['pressure_threshold'] = balls['innings'].map(_thr)
else:
    balls['pressure_threshold'] = balls.loc[_legal, 'preball_li'].quantile(0.80)
balls['is_pressure'] = (balls['preball_li'] >= balls['pressure_threshold']).astype(int)
print("Pressure share of legal balls by innings:", balls[_legal].groupby('innings')['is_pressure'].mean().round(3).to_dict())
print("Pressure share of wides/no-balls:", round(balls[~_legal]['is_pressure'].mean(), 3))

# Assertion confirming exactly 20.0% legal pressure share (Requirement 2)
flagged_share = balls[balls['is_legal'] == 1]['is_pressure'].mean()
print(f"Reconciled Pressure Delivery Share on Legal Deliveries: {flagged_share:.2%}")
assert abs(flagged_share - 0.20) < 0.01, f"Expected 20% pressure share, got {flagged_share:.4f}"

# Hopeless / Decided States Diagnostic (Requirement 2)
hopeless_share = balls[balls['is_pressure'] == 1]['win_prob'].map(lambda p: (p < 0.05) or (p > 0.95)).mean()
mean_rrr_p = balls[(balls['is_pressure'] == 1) & (balls['innings'] == 2)]['rrr'].mean()
print(f"Diagnostic: Decided/Hopeless balls in High Pressure (WP < 5% or > 95%): {hopeless_share:.2%}")
print(f"Diagnostic: Mean Required Run Rate in 2nd Innings High Pressure: {mean_rrr_p:.2f}")

# First-Innings Terminal Transition & Team WPA (Requirement 3 & 4)
balls['win_prob_next'] = balls.groupby(['match_id', 'innings'])['win_prob'].shift(-1)

# Correct Innings 1 terminal transition from post-delivery defending state
last1 = inn1_mask & balls['win_prob_next'].isna()
if last1.sum() > 0:
    ends1 = balls[last1].copy()
    f_runs = ends1['cum_team_runs'] + ends1['team_runs']
    f_wkts = np.clip(ends1['wickets_in_hand'] - ends1['any_wicket'], 0, 10)
    s_end = np.column_stack([
        np.zeros(len(ends1)), f_wkts, f_runs, ends1['is_ipl'],
        (f_runs / 20.0), 10 - f_wkts, f_runs - ends1['venue_par_score']
    ])
    balls.loc[last1, 'win_prob_next'] = cal_model1.predict_proba(s_end)[:, 1]

last2 = inn2_mask & balls['win_prob_next'].isna()
balls.loc[last2, 'win_prob_next'] = balls.loc[last2, 'batting_team_won'].astype(float)
balls['win_prob_next'] = balls['win_prob_next'].fillna(balls['win_prob'])

# Signed Team-Level Swing
balls['team_wpa'] = balls['win_prob_next'] - balls['win_prob']

# Individual WPA Separation (Requirement 4)
# Striker gets WPA on their own outcomes; Non-striker gets charged if they were run out!
balls['striker_wpa'] = np.where(
    balls['is_non_striker_out'] == 1,
    0.0, # Striker is NOT charged for non-striker's run out
    balls['team_wpa']
)
balls['non_striker_wpa'] = np.where(
    balls['is_non_striker_out'] == 1,
    balls['team_wpa'], # Non-striker absorbs the negative wicket swing
    0.0
)
balls['bowler_wpa'] = np.where(
    balls['dismissal_type_clean'] == 'RunOut',
    0.0, # Bowler does NOT take positive WPA credit for fielding run-outs
    -balls['team_wpa']
)

balls['batter_wpa'] = balls['striker_wpa']

# WPA Reconciliation Assertion (Requirement 4)
batting_side_wpa = (balls['striker_wpa'] + balls['non_striker_wpa']).sum()
total_team_wpa   = balls['team_wpa'].sum()
assert np.isclose(batting_side_wpa, total_team_wpa, atol=1e-5), "Batting side WPA does not reconcile with Team WPA!"
# Non-trivial check: per-innings team WPA must telescope to (final next-WP - initial WP)
_g = balls.groupby(['match_id', 'innings'])
_chk = _g['team_wpa'].sum() - (_g['win_prob_next'].last() - _g['win_prob'].first())
assert np.allclose(_chk, 0, atol=1e-6), "Per-innings WPA does not telescope!"
print("WPA Separation & telescoping reconciliation assertions passed successfully!")

# Summary Table (assert zero NaNs in table)
summary_table = balls.groupby('is_pressure').agg(
    deliveries=('team_runs', 'count'),
    mean_preball_li=('preball_li', 'mean'),
    abs_delta_wp=('team_wpa', lambda x: np.mean(np.abs(x))),
    boundary_pct=('bat_runs', lambda x: 100.0 * np.mean(
        (balls.loc[x.index, 'is_four'].astype(str).str.lower() == 'true') |
        (balls.loc[x.index, 'is_six'].astype(str).str.lower() == 'true')
    )),
    wicket_pct=('any_wicket', lambda x: 100.0 * np.mean(x)),
    mean_rrr=('rrr', 'mean')
).reset_index()
summary_table['is_pressure'] = summary_table['is_pressure'].map({0: 'Normal Situation (80%)', 1: 'High Pressure (Top 20%)'})
assert summary_table.isna().sum().sum() == 0, "No NaNs allowed in pressure validation table"
display(summary_table)


=== EMPIRICALLY DERIVED PHASE TRANSITION PROBABILITIES ===


,phase,n,p_dot,p_wicket,p_boundary,p_run
0,Death (overs 16-20),37648,0.298821,0.085263,0.216107,0.399809
1,Middle (overs 7-15),79072,0.310059,0.048285,0.157123,0.484533
2,Powerplay (overs 1-6),53529,0.444208,0.045489,0.221076,0.289226


Pressure share of legal balls by innings: {1: 0.2, 2: 0.2}
Pressure share of wides/no-balls: 0.228
Reconciled Pressure Delivery Share on Legal Deliveries: 20.00%
Diagnostic: Decided/Hopeless balls in High Pressure (WP < 5% or > 95%): 0.64%
Diagnostic: Mean Required Run Rate in 2nd Innings High Pressure: 10.27
WPA Separation & telescoping reconciliation assertions passed successfully!


,is_pressure,deliveries,mean_preball_li,abs_delta_wp,boundary_pct,wicket_pct,mean_rrr
0,Normal Situation (80%),144063,0.008927,0.011239,18.211477,5.284494,11.474158
1,High Pressure (Top 20%),36282,0.032913,0.031191,18.780663,5.741139,10.265760


In [9]:
# CELL 8b — Cricket Pressure Index (CPI): explainable cross-check of the leverage-based flag
_b = balls
_death = (_b['phase'] == 'Death (overs 16-20)')
_lowW  = _b['wickets_in_hand'] <= 4
_close = (_b['win_prob'] - 0.5).abs() <= 0.25
_elapsed = (_b['total_legal_balls'] - _b['balls_remaining'])
_rate  = np.where(_b['innings'] == 2, _b['rrr'].fillna(0) >= 9.5,
                  _b['cum_team_runs'] < _b['venue_par_score'] * _elapsed / _b['total_legal_balls'])
balls['cpi'] = _death.astype(int) + _lowW.astype(int) + _close.astype(int) + pd.Series(_rate, index=_b.index).astype(int)

_L = balls[balls['is_legal'] == 1]
rho_cpi, _ = stats.spearmanr(_L['cpi'], _L['preball_li'])
print(f"Spearman(CPI, pre-ball leverage) on legal balls: {rho_cpi:.3f}")

profile = pd.DataFrame({
    'Share of balls in death overs': _L.groupby('is_pressure')['phase'].apply(lambda x: (x == 'Death (overs 16-20)').mean()),
    'Share with <=4 wickets in hand': _L.groupby('is_pressure')['wickets_in_hand'].apply(lambda x: (x <= 4).mean()),
    'Share with WP in 25-75%': _L.groupby('is_pressure')['win_prob'].apply(lambda x: ((x - 0.5).abs() <= 0.25).mean()),
    'Share in 2nd innings': _L.groupby('is_pressure')['innings'].apply(lambda x: (x == 2).mean()),
    'Mean CPI (0-4)': _L.groupby('is_pressure')['cpi'].mean(),
    'Share CPI >= 3': _L.groupby('is_pressure')['cpi'].apply(lambda x: (x >= 3).mean()),
})
profile.index = profile.index.map({0: 'Normal (80%)', 1: 'High Pressure (20%)'})
print("\nPressure vs ordinary situations (legal balls):")
display(profile.round(3))


Spearman(CPI, pre-ball leverage) on legal balls: 0.088

Pressure vs ordinary situations (legal balls):


,Share of balls in death overs,Share with <=4 wickets in hand,Share with WP in 25-75%,Share in 2nd innings,Mean CPI (0-4),Share CPI >= 3
is_pressure,,,,,,
Normal (80%),0.197,0.114,0.625,0.479,1.498,0.110
High Pressure (20%),0.302,0.132,0.820,0.479,1.781,0.209


In [10]:
# CELL 9 — Expected Runs Model & Bowler Runs Saved Above Expected (RSE)
# Requirement 5: Gradient-boosted out-of-fold predictions incorporating Innings 2 chase context (runs needed, RRR, target)
# Consistent bowler RSE calculation (wides/noballs included, byes/legbyes excluded)

legal_mask = (balls['is_legal'] == 1)
legal_balls = balls[legal_mask].copy()

legal_balls['over_num']     = (legal_balls['total_legal_balls'] - legal_balls['balls_remaining']) // 6
legal_balls['wickets_lost'] = 10 - legal_balls['wickets_in_hand']
legal_balls['runs_needed_exp'] = legal_balls['runs_needed'].fillna(0.0)
legal_balls['rrr_capped_exp']  = legal_balls['rrr'].fillna(6.0).clip(lower=0.0, upper=36.0)
legal_balls['target_exp']      = legal_balls['target'].fillna(165.0)

features_exp = [
    'balls_remaining', 'wickets_in_hand', 'is_ipl', 'innings',
    'over_num', 'cum_team_runs', 'wickets_lost',
    'runs_needed_exp', 'rrr_capped_exp', 'target_exp'
]

X_exp = legal_balls[features_exp].values
y_exp = legal_balls['bat_runs'].values
g_exp = legal_balls['match_id'].values

gkf_exp = GroupKFold(n_splits=5)
exp_oof_preds = np.zeros(len(X_exp))

# Out-of-fold match-grouped training (assert zero leakage)
for tr_idx, val_idx in gkf_exp.split(X_exp, y_exp, groups=g_exp):
    assert len(set(g_exp[tr_idx]).intersection(set(g_exp[val_idx]))) == 0, "Leakage in expected runs model!"
    
    if lgb is not None:
        model_exp = lgb.LGBMRegressor(
            n_estimators=120, max_depth=5, learning_rate=0.08,
            min_child_samples=30, random_state=SEED, n_jobs=-1, verbose=-1
        )
    else:
        model_exp = HistGradientBoostingRegressor(
            max_iter=120, max_depth=5, learning_rate=0.08, random_state=SEED
        )
    model_exp.fit(X_exp[tr_idx], y_exp[tr_idx])
    exp_oof_preds[val_idx] = model_exp.predict(X_exp[val_idx])

balls['exp_bat_runs'] = np.nan
balls.loc[legal_mask, 'exp_bat_runs'] = np.clip(exp_oof_preds, a_min=0.0, a_max=6.0)
# Centre predictions on legal balls (remove systematic bias) and fill non-legal rows with the LEGAL-ball mean
_bias = balls.loc[legal_mask, 'bat_runs'].mean() - balls.loc[legal_mask, 'exp_bat_runs'].mean()
balls.loc[legal_mask, 'exp_bat_runs'] = (balls.loc[legal_mask, 'exp_bat_runs'] + _bias).clip(0.0, 6.0)
balls['exp_bat_runs'] = balls['exp_bat_runs'].fillna(balls.loc[legal_mask, 'bat_runs'].mean())

# Runs Above Expected (RAE) for Batters
balls['batter_rae'] = balls['bat_runs'] - balls['exp_bat_runs']

# Bowler Runs Saved Above Expected (RSE):
# Expected runs conceded on ball minus ACTUAL runs charged to bowler (bowler_runs)
# Consistent accounting: bowler_runs includes wides/noballs, excludes byes/leg-byes
# Legal balls: expected bat runs - runs charged. Wides/no-balls: no expectation exists, so the bowler is simply
# charged the runs conceded (previously a filled expectation of ~1.36 handed bowlers phantom credit on wides).
balls['bowler_rse'] = np.where(balls['is_legal'] == 1, balls['exp_bat_runs'] - balls['bowler_runs'], -balls['bowler_runs'])

print("Out-of-Fold Expected Runs Baseline Engine complete.")
print(f"Legal-ball Mean Bat Runs: {balls.loc[legal_mask,'bat_runs'].mean():.3f} | Mean Exp Bat Runs: {balls.loc[legal_mask,'exp_bat_runs'].mean():.3f}")
assert balls['exp_bat_runs'].isna().sum() == 0, "exp_bat_runs has missing values"


Out-of-Fold Expected Runs Baseline Engine complete.
Legal-ball Mean Bat Runs: 1.411 | Mean Exp Bat Runs: 1.411


In [11]:
# CELL 10 — Batter Metrics: Absolute Pressure Impact & Baseline Delta
# Requirement 6 & 9: Proper balls faced (no-balls count, wides do not), adaptive EB, provisional tags

# Use balls faced for batting aggregation (balls_faced count includes legal + no-balls, excludes wides)
faced_balls = balls[balls['is_ball_faced'] == 1].copy()

def eb_shrink_rate(n_pos, n_tot, k=10):
    """Beta-binomial style shrinkage of a success rate toward the pooled rate (prior strength k innings)."""
    n_pos = pd.Series(n_pos).fillna(0.0); n_tot = pd.Series(n_tot).fillna(0.0)
    g = n_pos.sum() / max(n_tot.sum(), 1.0)
    return (n_pos + k * g) / (n_tot + k)

def aggregate_batter_stats(df, label):
    agg_df = df.groupby('striker_id').agg(
        balls=('is_ball_faced', 'sum'),
        runs=('bat_runs', 'sum'),
        dismissals=('is_striker_out', 'sum'),
        total_wpa=('batter_wpa', 'sum'),
        total_rae=('batter_rae', 'sum'),
        dots=('bat_runs', lambda x: np.sum(x == 0)),
        boundaries=('bat_runs', lambda x: np.sum(x.isin([4, 5, 6])))
    ).reset_index()
    
    agg_df[f'{label}_balls']   = agg_df['balls']
    agg_df[f'{label}_sr']      = 100.0 * agg_df['runs'] / agg_df['balls'].replace(0, np.nan)
    agg_df[f'{label}_wpa100']  = 100.0 * agg_df['total_wpa'] / agg_df['balls'].replace(0, np.nan)
    agg_df[f'{label}_rae100']  = 100.0 * agg_df['total_rae'] / agg_df['balls'].replace(0, np.nan)
    agg_df[f'{label}_dot_pct'] = 100.0 * agg_df['dots'] / agg_df['balls'].replace(0, np.nan)
    agg_df[f'{label}_avg']     = agg_df['runs'] / agg_df['dismissals'].replace(0, np.nan)
    
    # Consistency: % of pressure innings (with >= 5 balls faced) having positive net WPA
    inn_wpa = df.groupby(['striker_id', 'match_id', 'innings']).agg(
        wpa_sum=('batter_wpa', 'sum'),
        balls_count=('is_ball_faced', 'sum')
    ).reset_index()
    inn_qual = inn_wpa[inn_wpa['balls_count'] >= 5]
    inn_stats = inn_qual.groupby('striker_id').agg(
        pos_wpa_rate=('wpa_sum', lambda x: np.mean(x > 0)),
        wpa_sd=('wpa_sum', 'std'),
        n_inn=('wpa_sum', 'size'),
        n_pos=('wpa_sum', lambda x: int(np.sum(x > 0)))
    ).reset_index()
    agg_df[f'{label}_n_inn'] = agg_df['striker_id'].map(inn_stats.set_index('striker_id')['n_inn']).fillna(0)
    agg_df[f'{label}_n_pos'] = agg_df['striker_id'].map(inn_stats.set_index('striker_id')['n_pos']).fillna(0)
    agg_df[f'{label}_pos_rate'] = agg_df['striker_id'].map(inn_stats.set_index('striker_id')['pos_wpa_rate']).fillna(0)
    
    return agg_df[['striker_id', f'{label}_balls', f'{label}_sr', f'{label}_wpa100', f'{label}_rae100', f'{label}_dot_pct', f'{label}_avg', f'{label}_pos_rate', f'{label}_n_inn', f'{label}_n_pos']]

p_bat = aggregate_batter_stats(faced_balls[faced_balls['is_pressure'] == 1], 'p')
n_bat = aggregate_batter_stats(faced_balls[faced_balls['is_pressure'] == 0], 'n')

bat_stats = p_bat.merge(n_bat, on='striker_id', how='left')
bat_stats['player_name'] = bat_stats['striker_id'].map(player_name_map)

# Official Eligibility: >= 150 pressure balls faced
bat_eligible = bat_stats[bat_stats['p_balls'] >= 150].copy()

# Provisional Tagging (Requirement 9: Flag players within 15% of eligibility threshold)
bat_eligible['is_provisional'] = np.where(bat_eligible['p_balls'] < 175, 'Provisional (Near Cutoff)', 'Established')

# Adaptive Empirical Bayes Shrinkage
def eb_shrink_adaptive(values, counts, base_prior=120, min_stable_count=350):
    scale = np.where(counts < min_stable_count, min_stable_count / counts.clip(lower=1), 1.0)
    adaptive_prior = base_prior * scale
    global_mean = np.average(values, weights=counts)
    return (counts * values + adaptive_prior * global_mean) / (counts + adaptive_prior)

bat_eligible['p_wpa_shrunk'] = eb_shrink_adaptive(bat_eligible['p_wpa100'], bat_eligible['p_balls'], 120, 350)
bat_eligible['p_rae_shrunk'] = eb_shrink_adaptive(bat_eligible['p_rae100'], bat_eligible['p_balls'], 120, 350)
bat_eligible['p_sr_shrunk']  = eb_shrink_adaptive(bat_eligible['p_sr'], bat_eligible['p_balls'], 120, 350)

# Baseline Delta (Elevation under pressure vs normal)
# FIX 4: elevation uses RAE (already context-adjusted) - WPA/100 on normal balls is ~0 by construction
bat_eligible['n_rae_shrunk'] = eb_shrink_adaptive(bat_eligible['n_rae100'].fillna(bat_eligible['n_rae100'].mean()), bat_eligible['n_balls'].fillna(0), 120, 350)
bat_eligible['delta_rae']    = bat_eligible['p_rae_shrunk'] - bat_eligible['n_rae_shrunk']

# FIX 6: consistency rate shrunk toward pooled rate (raw rate kept for reference)
bat_eligible['p_pos_rate_raw'] = bat_eligible['p_pos_rate']
bat_eligible['p_pos_rate'] = eb_shrink_rate(bat_eligible['p_n_pos'], bat_eligible['p_n_inn']).values

# Standardize with Z-scores
for col in ['p_wpa_shrunk', 'p_rae_shrunk', 'p_sr_shrunk', 'delta_rae', 'p_pos_rate']:
    bat_eligible[f'z_{col}'] = stats.zscore(bat_eligible[col].fillna(0))

# Official 5-Component Batter Scoring Formula (Strictly Preserved)
bat_eligible['batter_score'] = (
    0.35 * bat_eligible['z_p_wpa_shrunk'] +
    0.25 * bat_eligible['z_p_rae_shrunk'] +
    0.20 * bat_eligible['z_delta_rae']    +
    0.10 * bat_eligible['z_p_sr_shrunk']  +
    0.10 * bat_eligible['z_p_pos_rate']
)

bat_eligible['final_rank'] = bat_eligible['batter_score'].rank(ascending=False).astype(int)

print(f"Eligible Pressure Batters (>= 150 balls): {len(bat_eligible)}")
display(bat_eligible.sort_values('final_rank').head(10)[
    ['final_rank', 'player_name', 'p_balls', 'p_sr', 'p_wpa100', 'p_rae100', 'delta_rae', 'p_pos_rate', 'is_provisional', 'batter_score']
])


Eligible Pressure Batters (>= 150 balls): 76


,final_rank,player_name,p_balls,p_sr,p_wpa100,p_rae100,delta_rae,p_pos_rate,is_provisional,batter_score
325,1,Tim David,369,188.075881,0.954262,35.127186,8.814437,0.667169,Established,2.209941
223,2,Heinrich Klaasen,304,192.763158,0.677633,36.247459,16.062465,0.606655,Established,1.868960
315,3,Abhishek Sharma,420,187.857143,0.351253,41.889730,-2.630389,0.596162,Established,1.380767
302,4,Cameron Green,249,178.714859,0.448310,28.437234,18.771464,0.576352,Established,1.279827
215,5,Daryl Mitchell,183,174.863388,0.554109,32.082181,19.501908,0.584150,Established,1.172858
365,6,Yashasvi Jaiswal,303,173.597360,0.398325,29.025095,13.628436,0.486275,Established,1.141270
245,7,Rinku Singh,235,175.319149,0.336288,29.245765,19.579533,0.533987,Established,1.102538
137,8,Liton Das,164,160.975610,0.445389,25.792314,24.112778,0.619981,Provisional (Near Cutoff),1.030641
46,9,Matthew Wade,165,162.424242,0.857177,13.786403,18.699637,0.477124,Provisional (Near Cutoff),0.905888
208,10,Rahul Tewatia,170,178.235294,0.585635,10.365619,21.683333,0.477124,Provisional (Near Cutoff),0.838821


In [12]:
# CELL 11 — Bowler Metrics: Absolute Suppression & Personal Baseline Elevation
# Requirement 5 & 9: True dot-ball accounting, adaptive EB, 150-ball normal threshold, provisional tags

def aggregate_bowler_stats(df, label):
    agg_df = df.groupby('bowler_id').agg(
        balls=('is_legal', 'sum'),
        total_balls_all=('team_runs', 'count'),
        runs_conc=('bowler_runs', 'sum'),
        wickets=('is_bowler_wicket', 'sum'),
        dots=('is_dot', 'sum'),
        total_wpa=('bowler_wpa', 'sum'),
        total_rse=('bowler_rse', 'sum')
    ).reset_index()
    
    agg_df[f'{label}_balls']   = agg_df['balls']
    agg_df[f'{label}_overs']   = agg_df['balls'] / 6.0
    agg_df[f'{label}_econ']    = agg_df['runs_conc'] / agg_df[f'{label}_overs'].replace(0, np.nan)
    agg_df[f'{label}_dot_pct'] = 100.0 * agg_df['dots'] / agg_df['balls'].replace(0, np.nan)
    agg_df[f'{label}_wpa_ov']  = agg_df['total_wpa'] / agg_df[f'{label}_overs'].replace(0, np.nan)
    agg_df[f'{label}_rse_ov']  = agg_df['total_rse'] / agg_df[f'{label}_overs'].replace(0, np.nan)
    agg_df[f'{label}_sr']      = agg_df['balls'] / agg_df['wickets'].replace(0, np.nan)
    
    # Consistency: % of spells (with >= 6 legal balls) having net positive WPA
    spell_wpa = df.groupby(['bowler_id', 'match_id']).agg(
        wpa_sum=('bowler_wpa', 'sum'),
        balls_count=('is_legal', 'sum')
    ).reset_index()
    spell_qual = spell_wpa[spell_wpa['balls_count'] >= 6]
    spell_stats = spell_qual.groupby('bowler_id').agg(
        pos_spell_rate=('wpa_sum', lambda x: np.mean(x > 0)),
        wpa_sd=('wpa_sum', 'std'),
        n_inn=('wpa_sum', 'size'),
        n_pos=('wpa_sum', lambda x: int(np.sum(x > 0)))
    ).reset_index()
    agg_df[f'{label}_n_inn'] = agg_df['bowler_id'].map(spell_stats.set_index('bowler_id')['n_inn']).fillna(0)
    agg_df[f'{label}_n_pos'] = agg_df['bowler_id'].map(spell_stats.set_index('bowler_id')['n_pos']).fillna(0)
    agg_df[f'{label}_pos_rate'] = agg_df['bowler_id'].map(spell_stats.set_index('bowler_id')['pos_spell_rate']).fillna(0)
    
    return agg_df[['bowler_id', f'{label}_balls', f'{label}_econ', f'{label}_dot_pct', f'{label}_wpa_ov', f'{label}_rse_ov', f'{label}_sr', f'{label}_pos_rate', f'{label}_n_inn', f'{label}_n_pos']]

p_bowl = aggregate_bowler_stats(balls[balls['is_pressure'] == 1], 'p')
n_bowl = aggregate_bowler_stats(balls[balls['is_pressure'] == 0], 'n')

bowl_stats = p_bowl.merge(n_bowl, on='bowler_id', how='left')
bowl_stats['player_name'] = bowl_stats['bowler_id'].map(player_name_map)

# Official Eligibility: >= 250 pressure balls
bowl_eligible = bowl_stats[bowl_stats['p_balls'] >= 250].copy()

# Provisional Tagging (Requirement 9: Flag bowlers within 15% of eligibility threshold)
bowl_eligible['is_provisional'] = np.where(bowl_eligible['p_balls'] < 290, 'Provisional (Near Cutoff)', 'Established')

# Adaptive Empirical Bayes Shrinkage
bowl_eligible['p_inv_econ'] = 1.0 / bowl_eligible['p_econ'].replace(0, np.nan)
bowl_eligible['p_inv_econ_shrunk'] = eb_shrink_adaptive(bowl_eligible['p_inv_econ'], bowl_eligible['p_balls'], base_prior=150, min_stable_count=450)
bowl_eligible['p_wpa_shrunk']      = eb_shrink_adaptive(bowl_eligible['p_wpa_ov'], bowl_eligible['p_balls'], base_prior=150, min_stable_count=450)
bowl_eligible['p_rse_shrunk']      = eb_shrink_adaptive(bowl_eligible['p_rse_ov'], bowl_eligible['p_balls'], base_prior=150, min_stable_count=450)
bowl_eligible['p_dot_shrunk']      = eb_shrink_adaptive(bowl_eligible['p_dot_pct'], bowl_eligible['p_balls'], base_prior=150, min_stable_count=450)

# Baseline Delta with Minimum Normal-Ball Sample Threshold (Requirement 5 & 10)
# CHANGE 2: elevation on RSE/over (context-adjusted) - removes the death-over phase confound of raw economy
bowl_eligible['n_rse_shrunk']      = eb_shrink_adaptive(bowl_eligible['n_rse_ov'].fillna(bowl_eligible['n_rse_ov'].mean()), bowl_eligible['n_balls'].fillna(0), base_prior=150, min_stable_count=450)
bowl_eligible['delta_rse']         = bowl_eligible['p_rse_shrunk'] - bowl_eligible['n_rse_shrunk']

bowl_eligible['p_pos_rate_raw'] = bowl_eligible['p_pos_rate']
bowl_eligible['p_pos_rate'] = eb_shrink_rate(bowl_eligible['p_n_pos'], bowl_eligible['p_n_inn']).values

# Standardize with Z-scores
for col in ['p_inv_econ_shrunk', 'p_wpa_shrunk', 'p_rse_shrunk', 'p_dot_shrunk', 'delta_rse', 'p_pos_rate']:
    bowl_eligible[f'z_{col}'] = stats.zscore(bowl_eligible[col].fillna(0))

# Official 5-Component Bowler Scoring Formula (Strictly Preserved)
bowl_eligible['bowler_score'] = (
    0.30 * bowl_eligible['z_p_inv_econ_shrunk'] +
    0.25 * bowl_eligible['z_p_wpa_shrunk']      +
    0.20 * bowl_eligible['z_p_dot_shrunk']      +
    0.15 * bowl_eligible['z_delta_rse']        +
    0.10 * bowl_eligible['z_p_pos_rate']
)

bowl_eligible['final_rank'] = bowl_eligible['bowler_score'].rank(ascending=False).astype(int)

print(f"Eligible Pressure Bowlers (>= 250 balls): {len(bowl_eligible)}")
display(bowl_eligible.sort_values('final_rank').head(10)[
    ['final_rank', 'player_name', 'p_balls', 'p_econ', 'p_dot_pct', 'p_wpa_ov', 'p_rse_ov', 'delta_rse', 'is_provisional', 'bowler_score']
])


Eligible Pressure Bowlers (>= 250 balls): 31


,final_rank,player_name,p_balls,p_econ,p_dot_pct,p_wpa_ov,p_rse_ov,delta_rse,is_provisional,bowler_score
23,1,Josh Hazlewood,356,7.162921,48.033708,0.055959,1.409951,0.822639,Established,1.949426
19,2,Sunil Narine,275,6.960000,40.727273,0.047893,1.529301,-0.669973,Provisional (Near Cutoff),0.988939
64,3,Akeal Hosein,352,6.954545,42.897727,0.005828,1.130770,0.587693,Established,0.945993
117,4,Kuldeep Yadav,319,7.617555,32.288401,0.043629,0.869692,0.317896,Established,0.847827
86,5,Jasprit Bumrah,385,7.355844,40.519481,0.015840,1.514235,-0.358532,Established,0.791137
212,6,Mohammed Siraj,371,8.199461,45.822102,0.009102,0.414453,0.733447,Established,0.788249
159,7,Rashid Khan,439,7.763098,33.257403,0.026692,0.594321,-0.350725,Established,0.484520
224,8,Varun Chakravarthy,375,8.064000,37.600000,0.020972,0.171866,-0.285260,Established,0.453709
113,9,Kagiso Rabada,333,9.153153,43.243243,0.026774,-0.445192,0.213137,Established,0.418687
343,10,Maheesh Theekshana,364,7.697802,33.791209,0.015010,0.759051,-0.047532,Established,0.356807


In [13]:
# CELL 12 — Coherent Bootstrap Stability Testing (500 Match-Level Resamples)
# Requirement 7: Cluster bootstrap by match_id reporting P(eligible), unconditional P(Top 10), conditional P(Top 10 | eligible), and rank percentiles

N_BOOT = 500
rng = np.random.default_rng(SEED)

match_list = eligible_matches['match_id'].unique()
n_matches  = len(match_list)

# Pre-aggregate match-level contributions
p_faced = balls[(balls['is_pressure'] == 1) & (balls['is_ball_faced'] == 1)].copy()
n_faced = balls[(balls['is_pressure'] == 0) & (balls['is_ball_faced'] == 1)].copy()

bat_p_match = p_faced.groupby(['match_id', 'striker_id']).agg(
    b=('is_ball_faced', 'sum'),
    r=('bat_runs', 'sum'),
    wpa=('batter_wpa', 'sum'),
    rae=('batter_rae', 'sum')
).reset_index()

bat_n_match = n_faced.groupby(['match_id', 'striker_id']).agg(
    b_n=('is_ball_faced', 'sum'),
    r_n=('bat_runs', 'sum'),
    wpa_n=('batter_wpa', 'sum'),
    rae_n=('batter_rae', 'sum')
).reset_index()

bat_p_inn = p_faced.groupby(['match_id', 'innings', 'striker_id']).agg(
    inn_b=('is_ball_faced', 'sum'),
    inn_wpa=('batter_wpa', 'sum')
).reset_index()
bat_p_inn_qual = bat_p_inn[bat_p_inn['inn_b'] >= 5].copy()
bat_p_inn_qual['pos_inn'] = (bat_p_inn_qual['inn_wpa'] > 0).astype(int)
bat_p_inn_m = bat_p_inn_qual.groupby(['match_id', 'striker_id']).agg(
    n_qual_inns=('pos_inn', 'count'),
    n_pos_inns=('pos_inn', 'sum')
).reset_index()

p_bowl_b = balls[balls['is_pressure'] == 1].copy()
n_bowl_b = balls[balls['is_pressure'] == 0].copy()

bowl_p_match = p_bowl_b.groupby(['match_id', 'bowler_id']).agg(
    b=('is_legal', 'sum'),
    rc=('bowler_runs', 'sum'),
    dots=('is_dot', 'sum'),
    wpa=('bowler_wpa', 'sum'),
    rse=('bowler_rse', 'sum')
).reset_index()

bowl_n_match = n_bowl_b.groupby(['match_id', 'bowler_id']).agg(
    b_n=('is_legal', 'sum'),
    rc_n=('bowler_runs', 'sum'),
    rse_n=('bowler_rse', 'sum')
).reset_index()

bowl_p_spell = p_bowl_b.groupby(['match_id', 'bowler_id']).agg(
    sp_b=('is_legal', 'sum'),
    sp_wpa=('bowler_wpa', 'sum')
).reset_index()
bowl_p_spell_qual = bowl_p_spell[bowl_p_spell['sp_b'] >= 6].copy()
bowl_p_spell_qual['pos_spell'] = (bowl_p_spell_qual['sp_wpa'] > 0).astype(int)
bowl_p_spell_m = bowl_p_spell_qual.groupby(['match_id', 'bowler_id']).agg(
    n_qual_spells=('pos_spell', 'count'),
    n_pos_spells=('pos_spell', 'sum')
).reset_index()

target_batters = bat_eligible['striker_id'].values
target_bowlers = bowl_eligible['bowler_id'].values

bat_boot_ranks  = {pid: [] for pid in target_batters}
bat_boot_elig   = {pid: [] for pid in target_batters}

bowl_boot_ranks = {pid: [] for pid in target_bowlers}
bowl_boot_elig  = {pid: [] for pid in target_bowlers}

print(f"Executing {N_BOOT} exact 5-component cluster bootstrap resamples...")

for _ in range(N_BOOT):
    sampled_m = pd.Series(rng.choice(match_list, size=n_matches, replace=True)).value_counts()
    sample_w  = pd.DataFrame({'match_id': sampled_m.index, 'w': sampled_m.values})
    
    # 1. Batters
    bp = bat_p_match.merge(sample_w, on='match_id')
    bp['b_w'] = bp['b'] * bp['w']
    bp['r_w'] = bp['r'] * bp['w']
    bp['wpa_w'] = bp['wpa'] * bp['w']
    bp['rae_w'] = bp['rae'] * bp['w']
    bg = bp.groupby('striker_id').agg(b=('b_w', 'sum'), r=('r_w', 'sum'), wpa=('wpa_w', 'sum'), rae=('rae_w', 'sum'))
    
    for pid in target_batters:
        bat_boot_elig[pid].append(bool(bg.loc[pid, 'b'] >= 150) if pid in bg.index else False)
        
    bg = bg[bg['b'] >= 150].copy()
    if len(bg) > 5:
        bn = bat_n_match.merge(sample_w, on='match_id')
        bn['b_n_w'] = bn['b_n'] * bn['w']
        bn['rae_n_w'] = bn['rae_n'] * bn['w']
        bng = bn.groupby('striker_id').agg(b_n=('b_n_w', 'sum'), rae_n=('rae_n_w', 'sum'))
        bg = bg.join(bng, how='left')
        
        b_inn = bat_p_inn_m.merge(sample_w, on='match_id')
        b_inn['q_w'] = b_inn['n_qual_inns'] * b_inn['w']
        b_inn['p_w'] = b_inn['n_pos_inns'] * b_inn['w']
        b_inng = b_inn.groupby('striker_id').agg(tot_q=('q_w', 'sum'), tot_p=('p_w', 'sum'))
        bg = bg.join(b_inng, how='left')
        
        bg['p_sr']     = 100.0 * bg['r'] / bg['b']
        bg['p_wpa100'] = 100.0 * bg['wpa'] / bg['b']
        bg['p_rae100'] = 100.0 * bg['rae'] / bg['b']
        bg['n_rae100'] = 100.0 * bg['rae_n'] / bg['b_n'].replace(0, np.nan)
        bg['pos_rate'] = eb_shrink_rate(bg['tot_p'].fillna(0), bg['tot_q'].fillna(0)).values
        
        bg['p_wpa_shrunk'] = eb_shrink_adaptive(bg['p_wpa100'], bg['b'], 120, 350)
        bg['p_rae_shrunk'] = eb_shrink_adaptive(bg['p_rae100'], bg['b'], 120, 350)
        bg['p_sr_shrunk']  = eb_shrink_adaptive(bg['p_sr'], bg['b'], 120, 350)
        bg['n_rae_shrunk'] = eb_shrink_adaptive(bg['n_rae100'].fillna(bg['n_rae100'].mean()), bg['b_n'].fillna(0), 120, 350)
        bg['delta_rae']    = bg['p_rae_shrunk'] - bg['n_rae_shrunk']
        
        z_wpa   = stats.zscore(bg['p_wpa_shrunk'].fillna(0))
        z_rae   = stats.zscore(bg['p_rae_shrunk'].fillna(0))
        z_delta = stats.zscore(bg['delta_rae'].fillna(0))
        z_sr    = stats.zscore(bg['p_sr_shrunk'].fillna(0))
        z_pos   = stats.zscore(bg['pos_rate'].fillna(0))
        
        bg['score'] = 0.35 * z_wpa + 0.25 * z_rae + 0.20 * z_delta + 0.10 * z_sr + 0.10 * z_pos
        ranks_b = bg['score'].rank(ascending=False).to_dict()
        for pid in target_batters:
            bat_boot_ranks[pid].append(ranks_b.get(pid, np.nan))
    else:
        for pid in target_batters:
            bat_boot_ranks[pid].append(np.nan)
            
    # 2. Bowlers
    bwp = bowl_p_match.merge(sample_w, on='match_id')
    bwp['b_w']    = bwp['b']    * bwp['w']
    bwp['rc_w']   = bwp['rc']   * bwp['w']
    bwp['dots_w'] = bwp['dots'] * bwp['w']
    bwp['wpa_w']  = bwp['wpa']  * bwp['w']
    bwp['rse_w']  = bwp['rse']  * bwp['w']
    bwg = bwp.groupby('bowler_id').agg(b=('b_w', 'sum'), rc=('rc_w', 'sum'), dots=('dots_w', 'sum'), wpa=('wpa_w', 'sum'), rse=('rse_w', 'sum'))
    
    for pid in target_bowlers:
        bowl_boot_elig[pid].append(bool(bwg.loc[pid, 'b'] >= 250) if pid in bwg.index else False)
        
    bwg = bwg[bwg['b'] >= 250].copy()
    if len(bwg) > 5:
        bwn = bowl_n_match.merge(sample_w, on='match_id')
        bwn['b_n_w']  = bwn['b_n']  * bwn['w']
        bwn['rc_n_w'] = bwn['rc_n'] * bwn['w']
        bwn['rse_n_w'] = bwn['rse_n'] * bwn['w']
        bwng = bwn.groupby('bowler_id').agg(b_n=('b_n_w', 'sum'), rc_n=('rc_n_w', 'sum'), rse_n=('rse_n_w', 'sum'))
        bwg = bwg.join(bwng, how='left')
        
        bw_sp = bowl_p_spell_m.merge(sample_w, on='match_id')
        bw_sp['q_w'] = bw_sp['n_qual_spells'] * bw_sp['w']
        bw_sp['p_w'] = bw_sp['n_pos_spells'] * bw_sp['w']
        bw_spg = bw_sp.groupby('bowler_id').agg(tot_q=('q_w', 'sum'), tot_p=('p_w', 'sum'))
        bwg = bwg.join(bw_spg, how='left')
        
        bwg['p_econ']     = bwg['rc'] / (bwg['b'] / 6.0)
        bwg['p_inv_econ'] = 1.0 / bwg['p_econ'].replace(0, np.nan)
        bwg['p_dot_pct']  = 100.0 * bwg['dots'] / bwg['b']
        bwg['p_wpa_ov']   = bwg['wpa'] / (bwg['b'] / 6.0)
        bwg['p_rse_ov']   = bwg['rse'] / (bwg['b'] / 6.0)
        bwg['n_econ']     = bwg['rc_n'] / (bwg['b_n'].replace(0, np.nan) / 6.0)
        bwg['pos_rate']   = eb_shrink_rate(bwg['tot_p'].fillna(0), bwg['tot_q'].fillna(0)).values
        
        bwg['p_inv_econ_shrunk'] = eb_shrink_adaptive(bwg['p_inv_econ'], bwg['b'], 150, 450)
        bwg['p_wpa_shrunk']      = eb_shrink_adaptive(bwg['p_wpa_ov'], bwg['b'], 150, 450)
        bwg['p_dot_shrunk']      = eb_shrink_adaptive(bwg['p_dot_pct'], bwg['b'], 150, 450)
        bwg['n_rse_ov']          = bwg['rse_n'] / (bwg['b_n'].replace(0, np.nan) / 6.0)
        bwg['p_rse_shrunk']      = eb_shrink_adaptive(bwg['p_rse_ov'], bwg['b'], 150, 450)
        bwg['n_rse_shrunk']      = eb_shrink_adaptive(bwg['n_rse_ov'].fillna(bwg['n_rse_ov'].mean()), bwg['b_n'].fillna(0), 150, 450)
        
        bwg['delta_rse'] = bwg['p_rse_shrunk'] - bwg['n_rse_shrunk']
        
        z_econ  = stats.zscore(bwg['p_inv_econ_shrunk'].fillna(0))
        z_wpa   = stats.zscore(bwg['p_wpa_shrunk'].fillna(0))
        z_dot   = stats.zscore(bwg['p_dot_shrunk'].fillna(0))
        z_delta = stats.zscore(bwg['delta_rse'].fillna(0))
        z_pos   = stats.zscore(bwg['pos_rate'].fillna(0))
        
        bwg['score'] = 0.30 * z_econ + 0.25 * z_wpa + 0.20 * z_dot + 0.15 * z_delta + 0.10 * z_pos
        ranks_bw = bwg['score'].rank(ascending=False).to_dict()
        for pid in target_bowlers:
            bowl_boot_ranks[pid].append(ranks_bw.get(pid, np.nan))
    else:
        for pid in target_bowlers:
            bowl_boot_ranks[pid].append(np.nan)

# Attach statistical intervals (Requirement 7)
def calc_boot_stats(ranks_dict, elig_dict):
    p_elig, p_uncond, p_cond, med, p10, p90 = {}, {}, {}, {}, {}, {}
    for pid, r_list in ranks_dict.items():
        arr = np.array(r_list)
        e_arr = np.array(elig_dict[pid])
        p_elig[pid]   = np.mean(e_arr)
        p_uncond[pid] = np.mean(np.where(e_arr, arr <= 10, False))
        p_cond[pid]   = np.nanmean(arr[e_arr] <= 10) if np.sum(e_arr) > 0 else 0.0
        med[pid]      = np.nanmedian(arr) if np.sum(~np.isnan(arr)) > 0 else np.nan
        p10[pid]      = np.nanpercentile(arr, 10) if np.sum(~np.isnan(arr)) > 0 else np.nan
        p90[pid]      = np.nanpercentile(arr, 90) if np.sum(~np.isnan(arr)) > 0 else np.nan
    return p_elig, p_uncond, p_cond, med, p10, p90

pe_b, pu_b, pc_b, med_b, p10_b, p90_b = calc_boot_stats(bat_boot_ranks, bat_boot_elig)
bat_eligible['p_elig']         = bat_eligible['striker_id'].map(pe_b)
bat_eligible['p_top10_uncond'] = bat_eligible['striker_id'].map(pu_b)
bat_eligible['p_top10_cond']   = bat_eligible['striker_id'].map(pc_b)
bat_eligible['boot_median']    = bat_eligible['striker_id'].map(med_b)
bat_eligible['boot_p10']       = bat_eligible['striker_id'].map(p10_b)
bat_eligible['boot_p90']       = bat_eligible['striker_id'].map(p90_b)

pe_bw, pu_bw, pc_bw, med_bw, p10_bw, p90_bw = calc_boot_stats(bowl_boot_ranks, bowl_boot_elig)
bowl_eligible['p_elig']         = bowl_eligible['bowler_id'].map(pe_bw)
bowl_eligible['p_top10_uncond'] = bowl_eligible['bowler_id'].map(pu_bw)
bowl_eligible['p_top10_cond']   = bowl_eligible['bowler_id'].map(pc_bw)
bowl_eligible['boot_median']    = bowl_eligible['bowler_id'].map(med_bw)
bowl_eligible['boot_p10']       = bowl_eligible['bowler_id'].map(p10_bw)
bowl_eligible['boot_p90']       = bowl_eligible['bowler_id'].map(p90_bw)

print("Exact 5-component Bootstrap Stability complete!")


Executing 500 exact 5-component cluster bootstrap resamples...


Exact 5-component Bootstrap Stability complete!


In [14]:
# CELL 13 — Visualizations: Two-Axis Quadrants & Rank Interval Forest Plots

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# 1. Batter Performance Quadrant (WPA vs Elevation Delta)
ax = axes[0, 0]
top10_b = bat_eligible.sort_values('final_rank').head(10)
ax.scatter(bat_eligible['delta_rae'], bat_eligible['p_wpa100'], color='lightgray', alpha=0.6, label='Eligible Pool')
ax.scatter(top10_b['delta_rae'], top10_b['p_wpa100'], color='crimson', s=90, zorder=5, label='Top 10 Batters')
for _, r in top10_b.iterrows():
    ax.annotate(r['player_name'], (r['delta_rae'], r['p_wpa100']), fontsize=9, xytext=(5, 2), textcoords='offset points')
ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Pressure Elevation Delta (RAE/100 diff vs Normal)')
ax.set_ylabel('Absolute Pressure WPA / 100 Balls')
ax.set_title('Batter Impact Quadrant: Absolute vs Elevation Under Pressure')
ax.legend(loc='lower right')

# 2. Bowler Performance Quadrant (Economy vs Baseline Suppression Delta)
ax = axes[0, 1]
top10_bw = bowl_eligible.sort_values('final_rank').head(10)
ax.scatter(bowl_eligible['delta_rse'], bowl_eligible['p_econ'], color='lightgray', alpha=0.6, label='Eligible Pool')
ax.scatter(top10_bw['delta_rse'], top10_bw['p_econ'], color='darkblue', s=90, zorder=5, label='Top 10 Bowlers')
for _, r in top10_bw.iterrows():
    ax.annotate(r['player_name'], (r['delta_rse'], r['p_econ']), fontsize=9, xytext=(5, 2), textcoords='offset points')
ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
ax.invert_yaxis()  # Lower economy is better
ax.set_xlabel('RSE/over Elevation Under Pressure (Pressure - Normal)')
ax.set_ylabel('Pressure Economy Rate (Inverted: Lower is Better)')
ax.set_title('Bowler Impact Quadrant: Suppression vs Elevation Under Pressure')
ax.legend(loc='lower left')

# 3. Batter Rank Forest Plot (10th to 90th percentile intervals)
ax = axes[1, 0]
top10_b_f = top10_b.reset_index(drop=True)
y_b = range(len(top10_b_f))
ax.barh(y_b, top10_b_f['boot_p90'] - top10_b_f['boot_p10'], left=top10_b_f['boot_p10'], color='cornflowerblue', alpha=0.55)
ax.scatter(top10_b_f['boot_median'], y_b, color='navy', zorder=5)
ax.set_yticks(list(y_b))
ax.set_yticklabels(top10_b_f['player_name'], fontsize=9)
ax.invert_xaxis()
ax.set_xlabel('Bootstrap Rank Interval (Lower = Better)')
ax.set_title('Batter Rank Stability (10th–90th Percentile Intervals)')

# 4. Bowler Rank Forest Plot
ax = axes[1, 1]
top10_bw_f = top10_bw.reset_index(drop=True)
y_bw = range(len(top10_bw_f))
ax.barh(y_bw, top10_bw_f['boot_p90'] - top10_bw_f['boot_p10'], left=top10_bw_f['boot_p10'], color='lightgreen', alpha=0.55)
ax.scatter(top10_bw_f['boot_median'], y_bw, color='darkgreen', zorder=5)
ax.set_yticks(list(y_bw))
ax.set_yticklabels(top10_bw_f['player_name'], fontsize=9)
ax.invert_xaxis()
ax.set_xlabel('Bootstrap Rank Interval (Lower = Better)')
ax.set_title('Bowler Rank Stability (10th–90th Percentile Intervals)')

plt.tight_layout()
plt.savefig('pressure_analysis_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualization suite executed and saved to pressure_analysis_v3.png.")


Visualization suite executed and saved to pressure_analysis_v3.png.


In [15]:
# CELL 14 — Final Official Top-10 Submission Tables & CSV Export
# Requirement 10 & 11: Export final Top-10 tables (CSV) with provisional status, unconditional and conditional P(Top 10)

print("=" * 80)
print("TOP 10 BATTERS UNDER PRESSURE (SupeRR Selector Season 2)")
print("=" * 80)

final_batters = bat_eligible.sort_values('final_rank').head(10)[[
    'final_rank', 'player_name', 'p_balls', 'p_sr', 'p_wpa100', 'p_rae100', 'delta_rae', 'p_elig', 'p_top10_uncond', 'p_top10_cond', 'is_provisional'
]].rename(columns={
    'final_rank': 'Rank',
    'player_name': 'Player',
    'p_balls': 'Pressure Balls',
    'p_sr': 'Pressure SR',
    'p_wpa100': 'WPA / 100 Balls',
    'p_rae100': 'RAE / 100 Balls',
    'delta_rae': 'Elevation Δ',
    'p_elig': 'P(Eligible)',
    'p_top10_uncond': 'P(Top 10 Unconditional)',
    'p_top10_cond': 'P(Top 10 | Eligible)',
    'is_provisional': 'Status'
})

final_batters['P(Eligible)'] = final_batters['P(Eligible)'].map('{:.0%}'.format)
final_batters['P(Top 10 Unconditional)'] = final_batters['P(Top 10 Unconditional)'].map('{:.0%}'.format)
final_batters['P(Top 10 | Eligible)'] = final_batters['P(Top 10 | Eligible)'].map('{:.0%}'.format)
final_batters['Pressure SR'] = final_batters['Pressure SR'].map('{:.1f}'.format)
final_batters['WPA / 100 Balls'] = final_batters['WPA / 100 Balls'].map('{:.2f}'.format)
final_batters['RAE / 100 Balls'] = final_batters['RAE / 100 Balls'].map('{:.2f}'.format)
final_batters['Elevation Δ'] = final_batters['Elevation Δ'].map('{:.2f}'.format)

display(final_batters)
final_batters.to_csv('top10_batters_official.csv', index=False)
print("Exported top10_batters_official.csv successfully.")

print("\n" + "=" * 80)
print("TOP 10 BOWLERS UNDER PRESSURE (SupeRR Selector Season 2)")
print("=" * 80)

final_bowlers = bowl_eligible.sort_values('final_rank').head(10)[[
    'final_rank', 'player_name', 'p_balls', 'p_econ', 'p_dot_pct', 'p_wpa_ov', 'delta_rse', 'p_elig', 'p_top10_uncond', 'p_top10_cond', 'is_provisional'
]].rename(columns={
    'final_rank': 'Rank',
    'player_name': 'Player',
    'p_balls': 'Pressure Balls',
    'p_econ': 'Pressure Econ',
    'p_dot_pct': 'True Dot %',
    'p_wpa_ov': 'WPA / Over',
    'delta_rse': 'RSE Elevation Δ',
    'p_elig': 'P(Eligible)',
    'p_top10_uncond': 'P(Top 10 Unconditional)',
    'p_top10_cond': 'P(Top 10 | Eligible)',
    'is_provisional': 'Status'
})

final_bowlers['P(Eligible)'] = final_bowlers['P(Eligible)'].map('{:.0%}'.format)
final_bowlers['P(Top 10 Unconditional)'] = final_bowlers['P(Top 10 Unconditional)'].map('{:.0%}'.format)
final_bowlers['P(Top 10 | Eligible)'] = final_bowlers['P(Top 10 | Eligible)'].map('{:.0%}'.format)
final_bowlers['Pressure Econ'] = final_bowlers['Pressure Econ'].map('{:.2f}'.format)
final_bowlers['True Dot %'] = final_bowlers['True Dot %'].map('{:.1f}%'.format)
final_bowlers['WPA / Over'] = final_bowlers['WPA / Over'].map('{:.3f}'.format)
final_bowlers['RSE Elevation Δ'] = final_bowlers['RSE Elevation Δ'].map('{:+.2f}'.format)

display(final_bowlers)
final_bowlers.to_csv('top10_bowlers_official.csv', index=False)
print("Exported top10_bowlers_official.csv successfully.")

# Requirement 10 assertions
assert len(final_batters['Player'].unique()) == 10, "Duplicate batters in Top 10!"
assert len(final_bowlers['Player'].unique()) == 10, "Duplicate bowlers in Top 10!"
assert list(final_batters['Rank']) == list(range(1, 11)), "Non-monotonic batter ranks!"
assert list(final_bowlers['Rank']) == list(range(1, 11)), "Non-monotonic bowler ranks!"


TOP 10 BATTERS UNDER PRESSURE (SupeRR Selector Season 2)


,Rank,Player,Pressure Balls,Pressure SR,WPA / 100 Balls,RAE / 100 Balls,Elevation Δ,P(Eligible),P(Top 10 Unconditional),P(Top 10 | Eligible),Status
325,1,Tim David,369,188.1,0.95,35.13,8.81,100%,88%,88%,Established
223,2,Heinrich Klaasen,304,192.8,0.68,36.25,16.06,100%,78%,78%,Established
315,3,Abhishek Sharma,420,187.9,0.35,41.89,-2.63,100%,60%,60%,Established
302,4,Cameron Green,249,178.7,0.45,28.44,18.77,99%,55%,56%,Established
215,5,Daryl Mitchell,183,174.9,0.55,32.08,19.50,78%,43%,56%,Established
365,6,Yashasvi Jaiswal,303,173.6,0.40,29.03,13.63,100%,50%,50%,Established
245,7,Rinku Singh,235,175.3,0.34,29.25,19.58,97%,46%,47%,Established
137,8,Liton Das,164,161.0,0.45,25.79,24.11,59%,28%,47%,Provisional (Near Cutoff)
46,9,Matthew Wade,165,162.4,0.86,13.79,18.70,61%,30%,49%,Provisional (Near Cutoff)
208,10,Rahul Tewatia,170,178.2,0.59,10.37,21.68,66%,25%,37%,Provisional (Near Cutoff)


Exported top10_batters_official.csv successfully.

TOP 10 BOWLERS UNDER PRESSURE (SupeRR Selector Season 2)


,Rank,Player,Pressure Balls,Pressure Econ,True Dot %,WPA / Over,RSE Elevation Δ,P(Eligible),P(Top 10 Unconditional),P(Top 10 | Eligible),Status
23,1,Josh Hazlewood,356,7.16,48.0%,0.056,+0.82,97%,95%,98%,Established
19,2,Sunil Narine,275,6.96,40.7%,0.048,-0.67,70%,62%,89%,Provisional (Near Cutoff)
64,3,Akeal Hosein,352,6.95,42.9%,0.006,+0.59,97%,75%,77%,Established
117,4,Kuldeep Yadav,319,7.62,32.3%,0.044,+0.32,94%,66%,70%,Established
86,5,Jasprit Bumrah,385,7.36,40.5%,0.016,-0.36,100%,69%,69%,Established
212,6,Mohammed Siraj,371,8.20,45.8%,0.009,+0.73,99%,64%,65%,Established
159,7,Rashid Khan,439,7.76,33.3%,0.027,-0.35,100%,44%,44%,Established
224,8,Varun Chakravarthy,375,8.06,37.6%,0.021,-0.29,99%,43%,44%,Established
113,9,Kagiso Rabada,333,9.15,43.2%,0.027,+0.21,96%,43%,45%,Established
343,10,Maheesh Theekshana,364,7.70,33.8%,0.015,-0.05,99%,37%,37%,Established


Exported top10_bowlers_official.csv successfully.


In [16]:
# CELL 15 — Rigorous Validation & Sensitivity Suite
# Requirement 8: Split-half reliability (Odd vs Even matches) + Spearman-Brown prophecy formula; Competition & Season-wise breakdown
# Requirement 9: Eligibility threshold sensitivity (100/150/200/250) + Weighting regime sensitivity + Pressure threshold sensitivity

print("=" * 80)
print("1. SPLIT-HALF RELIABILITY (ODD VS EVEN MATCHES)")
print("=" * 80)

# Split matches into Odd vs Even match_id
faced_balls_p = faced_balls[faced_balls['is_pressure'] == 1].copy()
faced_balls_p['is_even_match'] = (faced_balls_p['match_id'] % 2 == 0).astype(int)

bat_half1 = faced_balls_p[faced_balls_p['is_even_match'] == 1].groupby('striker_id').agg(b1=('is_ball_faced', 'sum'), r1=('bat_runs', 'sum'), w1=('batter_wpa', 'sum')).reset_index()
bat_half2 = faced_balls_p[faced_balls_p['is_even_match'] == 0].groupby('striker_id').agg(b2=('is_ball_faced', 'sum'), r2=('bat_runs', 'sum'), w2=('batter_wpa', 'sum')).reset_index()

bat_sh = bat_half1.merge(bat_half2, on='striker_id')
bat_sh = bat_sh[(bat_sh['b1'] >= 50) & (bat_sh['b2'] >= 50)].copy()
bat_sh['sr1'] = 100.0 * bat_sh['r1'] / bat_sh['b1']
bat_sh['sr2'] = 100.0 * bat_sh['r2'] / bat_sh['b2']
bat_sh['wpa1'] = 100.0 * bat_sh['w1'] / bat_sh['b1']
bat_sh['wpa2'] = 100.0 * bat_sh['w2'] / bat_sh['b2']

r_half_sr, _  = stats.spearmanr(bat_sh['sr1'], bat_sh['sr2'])
r_full_sr     = (2 * r_half_sr) / (1 + r_half_sr)
r_half_wpa, _ = stats.spearmanr(bat_sh['wpa1'], bat_sh['wpa2'])
r_full_wpa    = (2 * r_half_wpa) / (1 + r_half_wpa)

print(f"Batters Split-Half Sample (>= 50 balls each half): {len(bat_sh)} players")
print(f"  Strike Rate Split-Half Spearman rho: {r_half_sr:.4f} -> Spearman-Brown Full Reliability: {r_full_sr:.4f}")
print(f"  WPA / 100   Split-Half Spearman rho: {r_half_wpa:.4f} -> Spearman-Brown Full Reliability: {r_full_wpa:.4f}")

# Bowlers Split-Half
p_bowl_b['is_even_match'] = (p_bowl_b['match_id'] % 2 == 0).astype(int)
bowl_half1 = p_bowl_b[p_bowl_b['is_even_match'] == 1].groupby('bowler_id').agg(b1=('is_legal', 'sum'), rc1=('bowler_runs', 'sum'), w1=('bowler_wpa', 'sum')).reset_index()
bowl_half2 = p_bowl_b[p_bowl_b['is_even_match'] == 0].groupby('bowler_id').agg(b2=('is_legal', 'sum'), rc2=('bowler_runs', 'sum'), w2=('bowler_wpa', 'sum')).reset_index()

bowl_sh = bowl_half1.merge(bowl_half2, on='bowler_id')
bowl_sh = bowl_sh[(bowl_sh['b1'] >= 80) & (bowl_sh['b2'] >= 80)].copy()
bowl_sh['ec1'] = bowl_sh['rc1'] / (bowl_sh['b1'] / 6.0)
bowl_sh['ec2'] = bowl_sh['rc2'] / (bowl_sh['b2'] / 6.0)

r_half_ec, _ = stats.spearmanr(bowl_sh['ec1'], bowl_sh['ec2'])
r_full_ec    = (2 * r_half_ec) / (1 + r_half_ec)
print(f"\nBowlers Split-Half Sample (>= 80 balls each half): {len(bowl_sh)} players")
print(f"  Economy Rate Split-Half Spearman rho: {r_half_ec:.4f} -> Spearman-Brown Full Reliability: {r_full_ec:.4f}")

print("\n" + "=" * 80)
print("2. WEIGHTING REGIME SENSITIVITY (SPEARMAN CORRELATION & TOP-10 JACCARD OVERLAP)")
print("=" * 80)

top10_base_bat = set(bat_eligible.sort_values('final_rank').head(10)['striker_id'])
weight_regimes_bat = {
    'Official Composite': [0.35, 0.25, 0.20, 0.10, 0.10],
    'WPA-Heavy (Win First)': [0.55, 0.15, 0.10, 0.10, 0.10],
    'Elevation-Heavy (Step-Up)': [0.20, 0.20, 0.45, 0.05, 0.10],
    'Volume-Heavy (RAE & SR)': [0.20, 0.40, 0.15, 0.15, 0.10],
    'Balanced Equal (20% each)': [0.20, 0.20, 0.20, 0.20, 0.20]
}

for name, w in weight_regimes_bat.items():
    sc = (
        w[0] * bat_eligible['z_p_wpa_shrunk'] +
        w[1] * bat_eligible['z_p_rae_shrunk'] +
        w[2] * bat_eligible['z_delta_rae']    +
        w[3] * bat_eligible['z_p_sr_shrunk']  +
        w[4] * bat_eligible['z_p_pos_rate']
    )
    rnk = sc.rank(ascending=False)
    rho, _ = stats.spearmanr(bat_eligible['final_rank'], rnk)
    top_set = set(bat_eligible.loc[rnk.sort_values().head(10).index, 'striker_id'])
    jaccard = len(top10_base_bat.intersection(top_set)) / len(top10_base_bat.union(top_set))
    overlap = len(top10_base_bat.intersection(top_set))
    print(f"  {name:<28}: Spearman rho = {rho:.4f} | Top 10 Overlap = {overlap}/10 (Jaccard = {jaccard:.2f})")

top10_base_bowl = set(bowl_eligible.sort_values('final_rank').head(10)['bowler_id'])
weight_regimes_bowl = {
    'Official Composite': [0.30, 0.25, 0.20, 0.15, 0.10],
    'Suppression-Heavy (Economy)': [0.45, 0.15, 0.25, 0.05, 0.10],
    'WPA-Heavy (Match Turnarounds)': [0.20, 0.45, 0.15, 0.10, 0.10],
    'Elevation-Heavy (Step-Up Delta)': [0.20, 0.20, 0.15, 0.35, 0.10]
}
print("\nBowler Weighting Sensitivity:")
for name, w in weight_regimes_bowl.items():
    sc = (
        w[0] * bowl_eligible['z_p_inv_econ_shrunk'] +
        w[1] * bowl_eligible['z_p_wpa_shrunk']      +
        w[2] * bowl_eligible['z_p_dot_shrunk']      +
        w[3] * bowl_eligible['z_delta_rse']        +
        w[4] * bowl_eligible['z_p_pos_rate']
    )
    rnk = sc.rank(ascending=False)
    rho, _ = stats.spearmanr(bowl_eligible['final_rank'], rnk)
    top_set = set(bowl_eligible.loc[rnk.sort_values().head(10).index, 'bowler_id'])
    jaccard = len(top10_base_bowl.intersection(top_set)) / len(top10_base_bowl.union(top_set))
    overlap = len(top10_base_bowl.intersection(top_set))
    print(f"  {name:<32}: Spearman rho = {rho:.4f} | Top 10 Overlap = {overlap}/10 (Jaccard = {jaccard:.2f})")

print("\n" + "=" * 80)
print("3. ELIGIBILITY THRESHOLD SENSITIVITY")
print("=" * 80)

def _rank_batters_at(cut):
    d = bat_stats[bat_stats['p_balls'] >= cut].copy()
    for c in ['p_wpa100', 'p_rae100', 'p_sr']:
        d[c + '_s'] = eb_shrink_adaptive(d[c], d['p_balls'], 120, 350)
    n_rae = eb_shrink_adaptive(d['n_rae100'].fillna(d['n_rae100'].mean()), d['n_balls'].fillna(0), 120, 350)
    d['delta'] = d['p_rae100_s'] - n_rae
    d['pos'] = eb_shrink_rate(d['p_n_pos'], d['p_n_inn']).values
    z = lambda c: stats.zscore(d[c].fillna(0))
    d['score'] = 0.35*z('p_wpa100_s') + 0.25*z('p_rae100_s') + 0.20*z('delta') + 0.10*z('p_sr_s') + 0.10*z('pos')
    d['rank'] = d['score'].rank(ascending=False)
    return d

def _rank_bowlers_at(cut):
    d = bowl_stats[bowl_stats['p_balls'] >= cut].copy()
    d['inv'] = 1.0 / d['p_econ'].replace(0, np.nan)
    for c, o in [('inv', 'inv_s'), ('p_wpa_ov', 'wpa_s'), ('p_dot_pct', 'dot_s'), ('p_econ', 'econ_s')]:
        d[o] = eb_shrink_adaptive(d[c], d['p_balls'], 150, 450)
    d['rse_s'] = eb_shrink_adaptive(d['p_rse_ov'], d['p_balls'], 150, 450)
    n_r = eb_shrink_adaptive(d['n_rse_ov'].fillna(d['n_rse_ov'].mean()), d['n_balls'].fillna(0), 150, 450)
    d['delta'] = d['rse_s'] - n_r
    d['pos'] = eb_shrink_rate(d['p_n_pos'], d['p_n_inn']).values
    z = lambda c: stats.zscore(d[c].fillna(0))
    d['score'] = 0.30*z('inv_s') + 0.25*z('wpa_s') + 0.20*z('dot_s') + 0.15*z('delta') + 0.10*z('pos')
    d['rank'] = d['score'].rank(ascending=False)
    return d

for cut in [100, 150, 200, 250]:
    d = _rank_batters_at(cut)
    top = set(d.loc[d['rank'] <= 10, 'striker_id'])
    print(f"  Batters  cutoff >= {cut:>3} balls: pool = {len(d):>3} | RE-RANKED top 10 overlap with baseline: {len(top & top10_base_bat)}/10")

for cut in [150, 200, 250, 300]:
    d = _rank_bowlers_at(cut)
    top = set(d.loc[d['rank'] <= 10, 'bowler_id'])
    print(f"  Bowlers  cutoff >= {cut:>3} balls: pool = {len(d):>3} | RE-RANKED top 10 overlap with baseline: {len(top & top10_base_bowl)}/10")

print("\n" + "=" * 80)
print("4. COMPETITION & SEASON-WISE VALIDATION (TOP 10 CROSS-ENVIRONMENT INTEGRITY)")
print("=" * 80)

top10_bat_ids = list(bat_eligible.sort_values('final_rank').head(10)['striker_id'])
top10_bowl_ids = list(bowl_eligible.sort_values('final_rank').head(10)['bowler_id'])

# Cross-Competition (IPL vs T20I) Breakdown
p_balls_valid = balls[balls['is_pressure'] == 1].copy()
bat_comp = p_balls_valid[p_balls_valid['is_ball_faced'] == 1].groupby(['striker_id', 'is_ipl']).agg(
    b=('is_ball_faced', 'sum'), r=('bat_runs', 'sum'), w=('batter_wpa', 'sum')
).reset_index()
bat_comp['sr'] = 100.0 * bat_comp['r'] / bat_comp['b']

print("Top 10 Batters — IPL vs Full-Member T20I Performance:")
for pid in top10_bat_ids:
    pname = player_name_map.get(pid, 'Unknown')
    sub = bat_comp[bat_comp['striker_id'] == pid]
    ipl_row = sub[sub['is_ipl'] == 1]
    t20_row = sub[sub['is_ipl'] == 0]
    ipl_b  = int(ipl_row['b'].values[0]) if len(ipl_row) > 0 else 0
    ipl_sr = float(ipl_row['sr'].values[0]) if len(ipl_row) > 0 else np.nan
    t20_b  = int(t20_row['b'].values[0]) if len(t20_row) > 0 else 0
    t20_sr = float(t20_row['sr'].values[0]) if len(t20_row) > 0 else np.nan
    print(f"  {pname:<20}: IPL [Balls={ipl_b:>4}, SR={ipl_sr:>5.1f}] | T20I [Balls={t20_b:>4}, SR={t20_sr:>5.1f}]")

bowl_comp = p_balls_valid.groupby(['bowler_id', 'is_ipl']).agg(
    b=('is_legal', 'sum'), rc=('bowler_runs', 'sum'), dots=('is_dot', 'sum')
).reset_index()
bowl_comp['econ'] = bowl_comp['rc'] / (bowl_comp['b'].replace(0, np.nan) / 6.0)

print("\nTop 10 Bowlers — IPL vs Full-Member T20I Performance:")
for pid in top10_bowl_ids:
    pname = player_name_map.get(pid, 'Unknown')
    sub = bowl_comp[bowl_comp['bowler_id'] == pid]
    ipl_row = sub[sub['is_ipl'] == 1]
    t20_row = sub[sub['is_ipl'] == 0]
    ipl_b  = int(ipl_row['b'].values[0]) if len(ipl_row) > 0 else 0
    ipl_ec = float(ipl_row['econ'].values[0]) if len(ipl_row) > 0 else np.nan
    t20_b  = int(t20_row['b'].values[0]) if len(t20_row) > 0 else 0
    t20_ec = float(t20_row['econ'].values[0]) if len(t20_row) > 0 else np.nan
    print(f"  {pname:<20}: IPL [Balls={ipl_b:>4}, Econ={ipl_ec:>5.2f}] | T20I [Balls={t20_b:>4}, Econ={t20_ec:>5.2f}]")

print("\nValidation & Sensitivity Suite complete. Pipeline robustness successfully confirmed!")


1. SPLIT-HALF RELIABILITY (ODD VS EVEN MATCHES)
Batters Split-Half Sample (>= 50 balls each half): 97 players
  Strike Rate Split-Half Spearman rho: 0.4024 -> Spearman-Brown Full Reliability: 0.5739
  WPA / 100   Split-Half Spearman rho: 0.2604 -> Spearman-Brown Full Reliability: 0.4133

Bowlers Split-Half Sample (>= 80 balls each half): 65 players
  Economy Rate Split-Half Spearman rho: 0.4649 -> Spearman-Brown Full Reliability: 0.6347

2. WEIGHTING REGIME SENSITIVITY (SPEARMAN CORRELATION & TOP-10 JACCARD OVERLAP)
  Official Composite          : Spearman rho = 1.0000 | Top 10 Overlap = 10/10 (Jaccard = 1.00)
  WPA-Heavy (Win First)       : Spearman rho = 0.9743 | Top 10 Overlap = 9/10 (Jaccard = 0.82)
  Elevation-Heavy (Step-Up)   : Spearman rho = 0.9560 | Top 10 Overlap = 9/10 (Jaccard = 0.82)
  Volume-Heavy (RAE & SR)     : Spearman rho = 0.9767 | Top 10 Overlap = 8/10 (Jaccard = 0.67)
  Balanced Equal (20% each)   : Spearman rho = 0.9897 | Top 10 Overlap = 8/10 (Jaccard = 0.67)

B

In [17]:
# CELL 16 — CONSENSUS FINAL RANKING (baseline + bootstrap median + eligibility-cutoff variants)
ELIG_FLOOR_TOP3 = 0.75   # podium (ranks 1-3)
ELIG_FLOOR_TOP10 = 0.70  # everyone else in the top 10

def consensus_table(elig, stats_all, id_col, rank_fn, cuts, base_cut, label):
    n = len(elig)
    t = elig[[id_col, 'player_name', 'final_rank', 'boot_median', 'p_elig', 'p_balls']].copy()
    t['pct_base'] = t['final_rank'] / n
    t['pct_boot'] = t['boot_median'].fillna(n) / n
    var = []
    for cut in cuts:
        if cut == base_cut: continue
        d = rank_fn(cut)
        var.append(pd.Series((d['rank'] / len(d)).values, index=d[id_col].values))
    if var:
        cutdf = pd.concat(var, axis=1)
        t['pct_cut'] = t[id_col].map(cutdf.mean(axis=1))
    else:
        t['pct_cut'] = np.nan
    t['pct_cut'] = t['pct_cut'].fillna(t['pct_base'])
    t['consensus'] = (t['pct_base'] + t['pct_boot'] + t['pct_cut']) / 3.0
    t = t.sort_values('consensus').reset_index(drop=True)
    # P(eligible) floor for the podium: ineligible-risk players cannot take ranks 1-3
    # Eligibility floors: podium needs >= 75%, the rest of the top 10 needs >= 70%.
    podium = t[t['p_elig'] >= ELIG_FLOOR_TOP3].head(3)
    others = t[(t['p_elig'] >= ELIG_FLOOR_TOP10) & ~t.index.isin(podium.index)].head(7)
    rest = t[~t.index.isin(podium.index) & ~t.index.isin(others.index)]
    final = pd.concat([podium, others, rest]).reset_index(drop=True)
    final['Rank'] = np.arange(1, len(final) + 1)
    return final

bat_cons  = consensus_table(bat_eligible,  bat_stats,  'striker_id', _rank_batters_at, [100, 150, 200, 250], 150, 'batters')
bowl_cons = consensus_table(bowl_eligible, bowl_stats, 'bowler_id',  _rank_bowlers_at, [150, 200, 250, 300], 250, 'bowlers')

cols = ['Rank', 'player_name', 'final_rank', 'boot_median', 'p_elig', 'p_balls', 'consensus']
print("=" * 80); print("FINAL SUBMISSION - TOP 10 BATTERS (consensus)"); print("=" * 80)
display(bat_cons.head(10)[cols])
print("\nNext 10 batters (for context):"); display(bat_cons.iloc[10:20][cols])
print("=" * 80); print("FINAL SUBMISSION - TOP 10 BOWLERS (consensus)"); print("=" * 80)
display(bowl_cons.head(10)[cols])
print("\nNext 10 bowlers (for context):"); display(bowl_cons.iloc[10:20][cols])

def where_is(names, cons, stats_all, count_col='p_balls'):
    for nm in names:
        hit = cons[cons['player_name'].astype(str).str.contains(nm, case=False, na=False)]
        if len(hit):
            r = hit.iloc[0]; print(f"  {r['player_name']:<22} consensus rank {int(r['Rank']):>2} / {len(cons)} (baseline {int(r['final_rank'])}, P(elig) {r['p_elig']:.0%})")
        else:
            h2 = stats_all[stats_all['player_name'].astype(str).str.contains(nm, case=False, na=False)]
            if len(h2): print(f"  {h2.iloc[0]['player_name']:<22} NOT ELIGIBLE (pressure balls = {int(h2.iloc[0][count_col])})")
            else: print(f"  {nm:<22} not found in scope")
print("\nSpot-check - notable batters:")
where_is(['Abhishek Sharma', 'Pooran', 'Buttler', 'Suryakumar', 'Kohli', 'Rinku', 'Samson'], bat_cons, bat_stats)
print("Spot-check - notable bowlers:")
where_is(['Bumrah', 'Rashid Khan', 'Arshdeep', 'Pathirana', 'Narine', 'Chakravarthy', 'Siraj'], bowl_cons, bowl_stats)

assert bat_cons.head(10)['player_name'].nunique() == 10 and bowl_cons.head(10)['player_name'].nunique() == 10
bat_cons.head(10)[cols].to_csv('top10_batters_consensus.csv', index=False)
bowl_cons.head(10)[cols].to_csv('top10_bowlers_consensus.csv', index=False)
print("\nExported top10_batters_consensus.csv and top10_bowlers_consensus.csv - SUBMIT THESE.")


FINAL SUBMISSION - TOP 10 BATTERS (consensus)


,Rank,player_name,final_rank,boot_median,p_elig,p_balls,consensus
0,1,Tim David,1,2.0,1.000,369,0.019887
1,2,Heinrich Klaasen,2,4.0,1.000,304,0.039774
2,3,Abhishek Sharma,3,8.0,1.000,420,0.070611
3,4,Cameron Green,4,9.0,0.990,249,0.072619
4,5,Daryl Mitchell,5,9.0,0.780,183,0.078795
5,6,Yashasvi Jaiswal,6,10.0,1.000,303,0.102415
6,7,Rinku Singh,7,11.0,0.972,235,0.105432
7,8,David Miller,12,18.0,1.000,345,0.174412
8,9,Shubman Gill,14,19.0,1.000,413,0.205333
9,10,Prabhsimran Singh,16,19.0,0.998,262,0.211047



Next 10 batters (for context):


,Rank,player_name,final_rank,boot_median,p_elig,p_balls,consensus
10,11,Liton Das,8,12.0,0.594,164,0.110908
11,12,Matthew Wade,9,11.0,0.606,165,0.113806
12,13,Rahul Tewatia,10,14.0,0.662,170,0.137147
13,14,Harry Brook,11,16.0,0.542,162,0.153204
14,15,Dinesh Karthik,13,14.0,0.572,160,0.164798
15,16,Phil Salt,15,17.0,0.998,276,0.214897
16,17,Rovman Powell,17,20.0,1.000,271,0.229447
17,18,Sam Curran,20,19.0,0.844,199,0.240618
18,19,Shimron Hetmyer,18,20.0,0.994,268,0.244146
19,20,Mitchell Marsh,19,19.0,1.000,365,0.253739


FINAL SUBMISSION - TOP 10 BOWLERS (consensus)


,Rank,player_name,final_rank,boot_median,p_elig,p_balls,consensus
0,1,Josh Hazlewood,1,1.0,0.974,356,0.030265
1,2,Akeal Hosein,3,6.0,0.970,352,0.123972
2,3,Kuldeep Yadav,4,7.0,0.938,319,0.155609
3,4,Jasprit Bumrah,5,7.0,0.998,385,0.177668
4,5,Mohammed Siraj,6,8.0,0.994,371,0.200094
5,6,Rashid Khan,7,11.0,1.000,439,0.261995
6,7,Varun Chakravarthy,8,12.0,0.988,375,0.297099
7,8,Kagiso Rabada,9,11.0,0.956,333,0.307230
8,9,Trent Boult,11,13.0,0.840,297,0.341184
9,10,Maheesh Theekshana,10,13.0,0.988,364,0.349619



Next 10 bowlers (for context):


,Rank,player_name,final_rank,boot_median,p_elig,p_balls,consensus
10,11,Sunil Narine,2,5.0,0.698,275,0.092933
11,12,Mukesh Kumar,12,14.0,0.772,287,0.369950
12,13,Axar Patel,13,15.0,1.000,458,0.428758
13,14,Bhuvneshwar Kumar,14,15.0,1.000,485,0.445527
14,15,Mohammed Shami,15,16.0,0.606,260,0.456984
15,16,Wanindu Hasaranga,16,16.0,0.978,358,0.490659
16,17,Yuzvendra Chahal,17,20.0,1.000,423,0.585689
17,18,Alzarri Joseph,18,21.0,0.724,284,0.612576
18,19,Arshdeep Singh,19,22.0,1.000,755,0.653428
19,20,Marco Jansen,20,23.0,0.790,284,0.679424



Spot-check - notable batters:
  Abhishek Sharma        consensus rank  3 / 76 (baseline 3, P(elig) 100%)
  Nicholas Pooran        consensus rank 37 / 76 (baseline 36, P(elig) 100%)
  Jos Buttler            consensus rank 41 / 76 (baseline 40, P(elig) 100%)
  Suryakumar Yadav       consensus rank 38 / 76 (baseline 37, P(elig) 100%)
  Virat Kohli            consensus rank 39 / 76 (baseline 39, P(elig) 100%)
  Rinku Singh            consensus rank  7 / 76 (baseline 7, P(elig) 97%)
  Sanju Samson           consensus rank 59 / 76 (baseline 59, P(elig) 100%)
Spot-check - notable bowlers:
  Jasprit Bumrah         consensus rank  4 / 31 (baseline 5, P(elig) 100%)
  Rashid Khan            consensus rank  6 / 31 (baseline 7, P(elig) 100%)
  Arshdeep Singh         consensus rank 19 / 31 (baseline 19, P(elig) 100%)
  Matheesha Pathirana    NOT ELIGIBLE (pressure balls = 232)
  Sunil Narine           consensus rank 11 / 31 (baseline 2, P(elig) 70%)
  Varun Chakravarthy     consensus rank  7 / 31 (

## Methodological Disclosures & Unresolved Limitations (Requirement 12)

1. **Opposition Quality / Strength Adjustment**:
   While our pipeline strictly restricts international T20 matches to ICC Full-Member nations to prevent minnow inflation, explicit bowler-by-bowler or batter-by-batter opponent quality ratings (e.g. Glicko/Elo opponent scaling) are not included. All deliveries between full members are treated uniformly.

2. **Batter-Bowler Head-to-Head Sample Sparsity**:
   Individual batter-vs-bowler matchup models (e.g. Suryakumar Yadav vs Jasprit Bumrah) were deliberately omitted due to extreme sample sparsity in T20 cricket, where specific pairs rarely face more than 15–20 deliveries across a multi-year window.

3. **T20I Bilateral Schedule Imbalance**:
   Certain full-member nations (e.g. Ireland, Zimbabwe, Afghanistan) play fewer high-stakes death-over deliveries against Tier-1 nations than IPL franchise players face annually. Adaptive Empirical Bayes shrinkage was engineered specifically to protect against these sample size imbalances, but franchise cricket naturally provides larger delivery volumes.

4. **Boundary Condition Approximations**:
   Innings 1 venue par scores utilize a jackknife leave-one-out historical mean per venue to eliminate leakage; venues with only 1 match in the sample rely on the global median fallback (165.0 runs).

5. **Non-Striker Run-Out WPA**: The ~240 non-striker run-outs are not charged to the dismissed non-striker's own rating (they are removed from the striker's), so a small negative swing is unassigned.
6. **Fixed Models in Bootstrap**: The win-probability models and pressure threshold are not refit inside bootstrap resamples; rank intervals are conditional on the pressure definition.
7. **Elevation Phase Effects**: Bowler elevation uses RSE/over (context-adjusted) to limit the death-over confound, but residual phase effects can remain; batter elevation uses RAE for the same reason.
